# 03b_Stage2_ML

## Stage B Introduction

Stage A of this project produced 20-quarter forecasts for 12 US macroeconomic variables.
Those forecasts answer the question of where is the economy likely to go? Stage B answers the
follow-on question: given where the economy is going, what does that imply for credit card
default rates?

Under IFRS 9 (International Financial Reporting Standard 9), banks must estimate Expected
Credit Losses using forward-looking macroeconomic information rather than historical
averages alone. Stage B builds the statistical link between the Stage A macroeconomic
forecasts and the credit card delinquency rate, translating economic conditions into a
predicted default rate.

The target variable is the US credit card loans delinquency rate (FRED series DRCCLACBS). Forecasting this rate over a
five-year horizon is a central deliverable of the project and the output that connects
the macroeconomic forecasting work in Stage A to the credit risk context BDO operates in.

The input file is `Stage1_final_regressors_US_Q.csv`, the output of notebook
`02c_Stage1_WinnerSelection`.

## How this notebook fits with the other Stage B notebook

Stage B is split across two notebooks by model family:

| Notebook | Owner | Model family |
|---|---|---|
| `03a_Stage2_TimeSeries` | Danny | ARMA, ARIMAX, SARIMAX |
| `03b_Stage2_ML` (this notebook) | Andres and Ashwin | Nine models, OLS through neural network |
| `03c_Stage2_WinnerSelection` | joint | Comparison of both tracks |

Both notebooks forecast the same target over the same evaluation window from the same
2020 Q4 origin, so the two model families are directly comparable in `03c`.

## Method in one paragraph

Regressors are chosen by a four-method consensus vote run on training data only. Nine
models are then fitted and compared over a single evaluation origin at 2020 Q4. OLS is
pre-registered as the primary model and the regulatory deliverable, a decision fixed before
any results were seen. The machine learning models are challengers. If a challenger wins,
that is reported, but OLS remains the deliverable because IFRS 9 model governance requires
coefficients that can be explained.

## Note on the primary evaluation metric

Root mean squared error (RMSE) is the primary metric for accuracy comparison and challenger
ranking in this notebook. This aligns Stage B with the Stage A metric set, where RMSE was
also primary, so that the two stages report accuracy on the same basis and the joint
comparison in `03c` is coherent. Mean absolute error is reported alongside RMSE throughout.
The change of primary metric does not affect the deliverable: OLS is the pre-registered
primary model under either metric. What it affects is which challenger is described as most
accurate, a secondary and exploratory claim.

## Structure of this notebook

| Section | Content |
|---|---|
| 0 | Imports and configuration |
| 1 | Data loading, validation and target visualisation |
| 2 | Joint regressor selection by four-method consensus vote |
| 3 | Feature matrix construction for the primary run and three robustness runs |
| 4 | Model fitting, nine models on a single-origin design |
| 5 | Evaluation, significance testing and SHAP attribution |
| 6 | Winner selection, production forecast to 2030 Q4 and export |

## References

Bank and Eder (2021) SSRN 3981339, IFRS 9 satellite model framework and probability of
default review.

Bellotti and Crook (2012) International Journal of Forecasting 28(1), OLS as benchmark on
credit card portfolios, VIF thresholds.

Bellotti and Crook (2013) International Journal of Forecasting 29(4), sign instability in
joint credit card default models.

Bergmeir, Hyndman and Koo (2018) Computational Statistics and Data Analysis 120, validity
of cross-validation for autocorrelated series.

Diebold and Mariano (1995) Journal of Business and Economic Statistics 13(3), testing
predictive accuracy.

Grinsztajn, Oyallon and Varoquaux (2022) NeurIPS 35, tree models against deep learning on
tabular data.

Lundberg and Lee (2017) NeurIPS 30, SHapley Additive Explanations.

Varma and Simon (2006) BMC Bioinformatics 7(91), bias in error estimation under model
selection.

## Section 0 - Imports and Configuration

All libraries and constants are defined in one cell so that no later cell introduces a
dependency or a magic number. Reading this cell first makes the rest of the notebook easier
to follow, since almost every variable name and date boundary used downstream appears here.

### Constants worth understanding before going further

`CANDIDATES` holds the 12 lagged regressor columns drawn from the Stage A output. Each
variable enters at the lag where its cross-correlation with the delinquency rate peaked in
the EDA (notebook 01, Section 5). The suffix records the lag, so `us_gdp_yoy_growth_L2`
means GDP growth two quarters earlier, reflecting the delay between an economic slowdown
and rising defaults. The 10-year bond yield enters as a first difference
(`us_bond_yield_10y_d1_L2`) because the level failed the EDA stationarity tests.

`CANDIDATES_R2` adds `us_cpi_L6`, a second CPI lag identified as a secondary peak in the
EDA. It competes only in robustness run R2 so that the primary specification keeps one lag
per variable.

`TRAIN_START` and `TRAIN_END` bound the training block, 1991 Q1 to 2020 Q4. Every model is
fitted here and the evaluation window is never shown to any model during fitting or tuning.

`TUNE_END`, `VAL_START` and `VAL_END` define the inner split used for hyperparameter
selection. The training block is divided into a tune period (1991 to 2016) and a validation
period (2017 to 2020). Candidate settings are fitted on the tune period and chosen on the
validation period. This is separate from the main evaluation and keeps tuning decisions away
from the test window.

`EVAL_START` and `EVAL_END` bound the test window, 2021 Q1 to 2025 Q4. Twenty quarters,
never seen during fitting.

`CLEAN_START` and `CLEAN_END` bound the clean sub-window, 2022 Q3 to 2025 Q4. The first six
quarters of the evaluation window overlap the COVID spline adjustment period, meaning the
target values there are reconstructed rather than observed. All primary performance claims
rest on the clean 14-quarter sub-window only.

`FCST_START` and `FCST_END` bound the production forecast horizon, 2026 Q1 to 2030 Q4.

`COVID_START` and `COVID_END` bound the distortion window, 2020 Q1 to 2022 Q2, over which
the target was reconstructed and the structural break dummy takes the value one.

`UNVALIDATED_STAGE_A` lists the six Stage A variables whose own univariate forecasts failed
the Diebold-Mariano test against naive persistence in notebook 02c. If any of these survive
the consensus vote, the forecast values they supply from 2026 onward carry more uncertainty
than the validated ones. This is tracked as a constant rather than a comment so that the
notebook can flag it automatically wherever it matters.

`SEED` fixes the random state for every stochastic component, so the notebook reproduces
exactly on re-run.

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.linear_model import Ridge, Lasso, ElasticNet, LassoCV
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.base import clone

import xgboost as xgb
from scipy import stats

SEED = 42

NAVY, TEAL, AMBER = '#1F3864', '#17A589', '#E67E22'
RED, GREEN, GREY  = '#C0392B', '#27AE60', '#7F8C8D'
BLUE, LBLUE       = '#2E75B6', '#AED6F1'
TEMPLATE = 'plotly_white'

DATA_DIR   = Path.cwd().parent / 'Data Collection'
LOCAL_FILE = DATA_DIR / 'Stage1_final_regressors_US_Q.csv'
GITHUB_RAW = 'https://raw.githubusercontent.com/hogandan85/ST-498/refs/heads/main/Data%20Collection'

REGRESSOR_FILE = (LOCAL_FILE if LOCAL_FILE.exists()
                  else f'{GITHUB_RAW}/Stage1_final_regressors_US_Q.csv')
OUT_DIR        = Path.cwd().parent / 'Stage2_Outputs'

TARGET     = 'us_delinquency_rate'
TARGET_RAW = 'us_delinquency_rate_raw'

TRAIN_START, TRAIN_END  = '1991-03-31', '2020-12-31'
TUNE_END                = '2016-12-31'
VAL_START,   VAL_END    = '2017-03-31', '2020-12-31'
EVAL_START,  EVAL_END   = '2021-03-31', '2025-12-31'
CLEAN_START, CLEAN_END  = '2022-09-30', '2025-12-31'
FCST_START,  FCST_END   = '2026-03-31', '2030-12-31'
COVID_START, COVID_END  = '2020-03-31', '2022-06-30'

HORIZONS = [1, 4, 8, 12, 20]

# Each variable at its peak CCF lag from 01_EDA Section 5.
# Bond yield enters as first difference: the level failed the EDA stationarity tests.
CANDIDATES = [
    'us_gdp_yoy_growth_L2',
    'us_unemployment_L0',
    'us_cpi_L0',
    'us_consumer_confidence_L2',
    'us_bond_yield_10y_d1_L2',
    'us_credit_qoq_growth_L6',
    'us_sp500_log_ret_L4',
    'us_vix_log_ret_L0',
    'us_house_price_yoy_L3',
    'us_indprod_yoy_L3',
    'us_oil_yoy_L2',
    'us_reer_diff_L0',
]

# CPI L6 competes only in the lag-rich robustness run (R2), not the primary vote.
CANDIDATES_R2 = CANDIDATES + ['us_cpi_L6']

# Stage A winners that failed the DM test against naive persistence (02c).
UNVALIDATED_STAGE_A = [
    'us_consumer_confidence', 'us_credit_qoq_growth', 'us_indprod_yoy',
    'us_reer_diff', 'us_sp500_log_ret', 'us_vix_log_ret',
]

print(f'Configuration loaded | {len(CANDIDATES)} primary candidates | seed {SEED}')
print(f'Input: {"local" if LOCAL_FILE.exists() else "GitHub"} | {REGRESSOR_FILE}')
print(f'Output directory: {OUT_DIR}')

Configuration loaded | 12 primary candidates | seed 42
Input: local | c:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Data Collection\Stage1_final_regressors_US_Q.csv
Output directory: c:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Stage2_Outputs


## Section 1 - Data Loading and Validation

### Input file

`Stage1_final_regressors_US_Q.csv` is the final output of the Stage A winner selection
process in notebook 02c. It holds 164 quarterly rows covering 1990 Q1 to 2030 Q4. Rows up
to 2025 Q4 are historical observations. Rows from 2026 Q1 onward are the Stage A
macroeconomic forecasts that Stage B uses to project the delinquency rate forward.

The loading cell raises an error rather than printing a warning if a required column is
missing or the row count is not 164. A stale or partially written input file would otherwise
propagate silently into every result downstream.

### Why does the target variable have two versions?

During 2020 Q1 to 2022 Q2, government stimulus payments and loan moratoria held credit card
delinquency well below where macroeconomic conditions alone would have put it. The EBA
(2021, paragraph 75) documents this directly - public support measures reduced the level of
defaults that would otherwise have been observed.

A model trained on those artificially low values learns a weaker relationship between the
economy and default rates than genuinely holds. It would then underestimate default risk in
future stress scenarios, which is the opposite of what IFRS 9 is designed to achieve.

To correct this, a cubic spline was fitted through the surrounding non-COVID observations in
the EDA notebook and used to reconstruct a counterfactual path across the distortion window.
That adjusted series is `us_delinquency_rate` and is the modelling target throughout. The
original observed series is retained as `us_delinquency_rate_raw` and used only in
robustness run R3, which tests whether the adjustment matters.

The adjustment is contested. Liu et al. (2025) take the opposite position and exclude the
distorted quarters entirely rather than reconstructing them. R3 implements their approach so
the two can be compared directly.

### Why do usable quarters fall short of calendar quarters?

A quarter is usable only if the target and every regressor in the specification are all
observed. Two things reduce the count. Lagged variables lose their leading quarters, since a
lag-6 column has no value until six quarters of the base series exist. And the real effective
exchange rate has no data before 1994 Q2.

Requiring all 12 candidates therefore costs 13 of the 120 calendar quarters in the training
block, and the usable sample begins in 1994 Q2 rather than 1991 Q1. Table 1.2 shows which
variables bind. The usable sample size depends on the feature set, so the figures quoted in
the report come from these printed counts rather than from the calendar length.

### Figure 1.1

The figure plots both target series with three regions marked: the COVID reconstruction
window in red, the clean evaluation sub-window in blue, and the evaluation origin as a
dashed vertical line. The six quarters between the dashed line and the blue band are the
evaluation quarters scored against reconstructed values, which is why the clean sub-window
exists as a separate reporting window.

In [2]:
df = pd.read_csv(REGRESSOR_FILE, index_col=0, parse_dates=True).sort_index()

required = CANDIDATES + [TARGET, TARGET_RAW, 'covid_dummy']
missing  = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f'Input file missing required columns: {missing}')
if len(df) != 164:
    raise ValueError(f'Expected 164 rows from 02c, found {len(df)}')

print(f'Loaded {df.shape[0]} rows x {df.shape[1]} columns '
      f'({df.index.min().date()} to {df.index.max().date()})')
print(f'All {len(required)} required columns present.')
if 'delinquency_spline' in df.columns:
    print('NOTE: delinquency_spline still present - 02c drop not applied in this version.')

df_train = df.loc[TRAIN_START:TRAIN_END].copy()
df_tune  = df.loc[TRAIN_START:TUNE_END].copy()
df_val   = df.loc[VAL_START:VAL_END].copy()
df_eval  = df.loc[EVAL_START:EVAL_END].copy()
df_clean = df.loc[CLEAN_START:CLEAN_END].copy()
df_full  = df.loc[TRAIN_START:EVAL_END].copy()
df_fcst  = df.loc[FCST_START:FCST_END].copy()


def q(ts):
    return f'{ts.year} Q{ts.quarter}'

blocks = [('Training block', df_train), ('Tune split', df_tune), ('Val split', df_val),
          ('Evaluation', df_eval), ('Clean sub-window', df_clean), ('Full panel', df_full)]

print('\nTime splits (calendar quarters | usable with all 12 candidates):')
for name, b in blocks:
    n_cal = len(b)
    n_use = len(b[CANDIDATES + [TARGET]].dropna())
    print(f'  {name:<17}: {q(b.index.min())} to {q(b.index.max())}   {n_cal:>3} | {n_use:>3}')

print(f'  {"Forecast":<17}: {q(df_fcst.index.min())} to {q(df_fcst.index.max())}   '
      f'{len(df_fcst):>3} | {len(df_fcst[CANDIDATES].dropna()):>3}')

Loaded 164 rows x 42 columns (1990-03-31 to 2030-12-31)
All 15 required columns present.

Time splits (calendar quarters | usable with all 12 candidates):
  Training block   : 1991 Q1 to 2020 Q4   120 | 107
  Tune split       : 1991 Q1 to 2016 Q4   104 |  91
  Val split        : 2017 Q1 to 2020 Q4    16 |  16
  Evaluation       : 2021 Q1 to 2025 Q4    20 |  20
  Clean sub-window : 2022 Q3 to 2025 Q4    14 |  14
  Full panel       : 1991 Q1 to 2025 Q4   140 | 127
  Forecast         : 2026 Q1 to 2030 Q4    20 |  20


In [3]:
starts = pd.DataFrame([
    {'Variable': c,
     'First valid': df[c].first_valid_index().date(),
     'NaNs in training block': int(df_train[c].isna().sum())}
    for c in CANDIDATES + [TARGET]
]).sort_values('First valid', ascending=False)

print('Table 1.2 - First valid observation per candidate')
display(starts.set_index('Variable'))

binding = df_train[CANDIDATES + [TARGET]].isna().any(axis=1)
print(f'Training rows lost to listwise deletion: {binding.sum()} of {len(df_train)}')
print(f'First complete case: {df_train[~binding].index.min().date()}')

Table 1.2 - First valid observation per candidate


,First valid,NaNs in training block
Variable,,
us_reer_diff_L0,1994-06-30,13
us_house_price_yoy_L3,1991-12-31,3
us_indprod_yoy_L3,1991-12-31,3
us_credit_qoq_growth_L6,1991-09-30,2
us_oil_yoy_L2,1991-09-30,2
us_sp500_log_ret_L4,1991-03-31,0
us_delinquency_rate,1991-03-31,0
us_bond_yield_10y_d1_L2,1990-12-31,0
us_gdp_yoy_growth_L2,1990-09-30,0


Training rows lost to listwise deletion: 13 of 120
First complete case: 1994-06-30


In [4]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_full.index, y=df_full[TARGET_RAW],
    mode='lines', name='Raw observed',
    line=dict(color=NAVY, width=2)))

fig.add_trace(go.Scatter(
    x=df_full.index, y=df_full[TARGET],
    mode='lines', name='Spline-adjusted (model target)',
    line=dict(color=AMBER, width=2, dash='dot')))

# COVID spline window: target is reconstructed, not observed
fig.add_vrect(
    x0=COVID_START, x1=COVID_END,
    fillcolor='rgba(192,57,43,0.10)', line_width=0,
    annotation_text='COVID spline window', annotation_position='top left',
    annotation_font=dict(size=9, color=RED))

# Clean sub-window: evaluation quarters scored against genuinely observed values
fig.add_vrect(
    x0=CLEAN_START, x1=CLEAN_END,
    fillcolor='rgba(46,117,182,0.08)', line_width=0,
    annotation_text='Clean sub-window', annotation_position='right',
    annotation_font=dict(size=9, color=BLUE))

# Evaluation origin. Annotation added separately: add_vline's own annotation breaks on
# date-string axes in older plotly (it averages the x coords to place the label).
fig.add_vline(x=EVAL_START, line_dash='dash', line_color=GREY, line_width=1.2)
fig.add_annotation(
    x=EVAL_START, y=0.02, yref='paper', text='Evaluation origin',
    showarrow=False, font=dict(size=9, color=GREY),
    xanchor='left', xshift=4, bgcolor='rgba(255,255,255,0.7)')

train_mean = df_train[TARGET].mean()
fig.add_hline(y=train_mean, line_dash='dot', line_color=GREY, line_width=1,
              annotation_text=f'Training mean = {train_mean:.2f}%',
              annotation_position='bottom right',
              annotation_font=dict(size=9, color=GREY))

fig.update_layout(
    title=dict(
        text=('<b>Figure 1.1 - Target Variable: Raw vs Spline-Adjusted</b>'
              f'<br><span style="font-size:11.5px;color:{GREY}">'
              'Red = spline reconstruction window  (2020Q1-2022Q2) | Blue = clean evaluation sub-window  (2022Q3-2025Q4) | '
              'the 6 quarters between them are scored against imputed values</span>'),
        font=dict(size=16, color=NAVY), x=0.015, xanchor='left', y=0.96, yanchor='top'),
    xaxis_title='Quarter', yaxis_title='Delinquency Rate (%)',
    template=TEMPLATE, height=480,
    legend=dict(orientation='h', yanchor='top', y=1.15, xanchor='center', x=0.5),
    margin=dict(t=115, b=55, l=70, r=30))
fig.show()

## Section 2 - Joint Regressor Selection

### Replacing the cross-correlation screen with a consensus vote

The earlier approach in the report selected regressors on whether their univariate
cross-correlation with the delinquency rate cleared a 95 per cent significance threshold.
That approach has a known weakness. A variable can correlate strongly with the target on its
own yet become redundant, or change sign, once correlated variables enter alongside it.

Bellotti and Crook (2013, Tables 3 and 4) document exactly this for house prices in a credit
card default model - a variable that looks significant univariately can have its sign flip or
its coefficient collapse when entered jointly. Screening univariately and then fitting a
joint model is therefore internally inconsistent.

Running selection inside a multivariate framework avoids the problem. But every individual
selection method has its own blind spot -
- Backward stepwise regression is sensitive to the order in which variables are removed.
- Lasso chooses arbitrarily between correlated variables and can return different subsets under small data perturbations.
- Tree importance scores favour variables with more unique values or stronger nonlinear effects.

Requiring a variable to survive at least two of four independent methods reduces the chance that any single
method's idiosyncrasy determines the final specification (Liu et al. 2025, Section 4).

### The four methods

Backward stepwise by AIC starts with all 12 candidates and removes the variable whose
removal most improves the Akaike Information Criterion, stopping when no removal helps or
when five remain. AIC balances fit against the number of parameters, so it penalises
variables that add little.

Lasso fits a penalised regression where the L1 penalty shrinks coefficients toward zero. The
penalty strength is chosen by time-series cross-validation. Variables are then ranked by the
absolute size of their standardised coefficient.

Random Forest permutation importance measures how much predictive accuracy drops when a
variable's values are randomly shuffled. A variable the model relies on produces a large
drop, an irrelevant one produces none.

Gradient boosting relative importance ranks variables by how much they contribute to
reducing training error across all trees in the ensemble.

Each method nominates its top five of the 12 candidates.

### Why training data only?

Information leakage is the most common way an out-of-sample evaluation becomes misleadingly
optimistic. If selection ran on the full dataset including 2021 to 2025, the model would have
indirectly seen the test window and chosen variables partly because they happened to work
there. The evaluation would no longer be genuinely out of sample. All four methods therefore
run on the training block only (199x-2020).

### Why is the COVID dummy forced in rather than voted on?

The dummy is a structural break control, not a macroeconomic predictor. Its role is to absorb
the level shift caused by policy intervention so that the macroeconomic coefficients are not
distorted by it. Letting it compete in the vote would confuse two different jobs - explaining
economic transmission, and controlling for a known one-off distortion. It is therefore forced
into every specification and never treated as a candidate.

### Rules fixed before the vote was run

Selection runs on the training block only. The dummy is forced in. CPI competes on the same
terms as every other variable rather than being retained by assumption. Each method nominates
five. A variable is retained at two or more votes. If the vote had returned fewer than four
or more than eight variables, a pre-specified tie-break would take the top six by vote count
and then by mean rank across methods.

The retained set defines Equation 5.11 (exact number is tentative) in the report. It is a result reported in Section 6,
not a methodology pre-specification.

### Sensitivity checks

Two cells test whether the outcome depends on choices that could have gone another way.

The first re-runs the vote without the real effective exchange rate. That variable has no
data before 1994 Q2 and truncates the selection sample by 13 quarters, so the justification
for testing its exclusion is data availability, which is knowable without reference to any
vote outcome. This matters as excluding a variable because it lost the vote would be
result-dependent selection, whereas excluding it because it costs 13 observations is not.

The second varies how many variables each method nominates, from four to seven. If the
retained set is stable across those widths, the choice of five was not decisive.

### Diagnostics

The remaining cells in this section feed no selection decision.

All coefficient comparisons run on one fixed sample, held in `COMMON_IDX`. Without this, a
variable with a shorter history changes both the specification and the set of quarters
included at the same time, and the two effects cannot be separated. This is the same
listwise-deletion trap that Table 1.2 documents, appearing again at the diagnostic stage.

Table 2.2 checks coefficient signs against economic priors, fitted both on all 12 candidates
and on the selected set, so that a stable sign can be distinguished from one that moves.

Table 2.3 reports variance inflation factors on the model's own sample rather than
`COMMON_IDX`, since variance inflation should describe the model actually fitted. The
thresholds come from Bellotti and Crook (2012, Section 3.2): below 5 for OLS, below 10 for
regularised and nonlinear models, which tolerate correlated inputs better.

The final cell traces one coefficient across nested specifications to locate the point at
which its sign changes.

In [5]:
# 2.1 - selection sample and vote parameters
train_sel = df_train[CANDIDATES + [TARGET, 'covid_dummy']].dropna()

X_sel = train_sel[CANDIDATES]
y_sel = train_sel[TARGET].values

# Scaled copy shared by the Lasso and tree methods. Built once here so the four
# selection cells below can run in any order.
scaler_sel = StandardScaler()
X_sel_s    = scaler_sel.fit_transform(X_sel)

TOP_N          = 5   # each method nominates this many
VOTE_THRESHOLD = 2   # retained if nominated by at least this many methods

print(f'Selection sample: {len(train_sel)} quarters '
      f'({q(train_sel.index.min())} to {q(train_sel.index.max())})')
print(f'{len(CANDIDATES)} candidates | each method nominates {TOP_N} | '
      f'retained at >= {VOTE_THRESHOLD} of 4 votes')

Selection sample: 107 quarters (1994 Q2 to 2020 Q4)
12 candidates | each method nominates 5 | retained at >= 2 of 4 votes


In [6]:
# 2.2 - method 1 backward stepwise AIC
def backward_stepwise_aic(X_df, y, feature_cols, top_n):
    """Backward elimination by AIC. Starts with all candidates and drops the variable
    whose removal most improves AIC, stopping at top_n or when no removal helps."""
    remaining = list(feature_cols)
    while len(remaining) > top_n:
        full_aic = sm.OLS(y, sm.add_constant(X_df[remaining].values)).fit().aic
        best_aic, to_remove = full_aic, None
        for feat in remaining:
            others = [f for f in remaining if f != feat]
            aic = sm.OLS(y, sm.add_constant(X_df[others].values)).fit().aic
            if aic < best_aic:
                best_aic, to_remove = aic, feat
        if to_remove is None:
            break
        remaining.remove(to_remove)
    return remaining[:top_n]

top5_aic = backward_stepwise_aic(X_sel, y_sel, CANDIDATES, TOP_N)

print(f'Backward AIC top {len(top5_aic)}:')
for v in top5_aic:
    print(f'  {v}')

Backward AIC top 5:
  us_gdp_yoy_growth_L2
  us_unemployment_L0
  us_cpi_L0
  us_consumer_confidence_L2
  us_credit_qoq_growth_L6


In [7]:
# 2.3 - method 2 Lasso
lasso_cv = LassoCV(cv=TimeSeriesSplit(n_splits=5), max_iter=10000, random_state=SEED)
lasso_cv.fit(X_sel_s, y_sel)

lasso_ranked = sorted(
    [(c, abs(w)) for c, w in zip(CANDIDATES, lasso_cv.coef_) if abs(w) > 1e-6],
    key=lambda t: t[1], reverse=True)
top5_lasso = [c for c, _ in lasso_ranked[:TOP_N]]

print(f'Lasso (alpha={lasso_cv.alpha_:.4f}) nominated {len(lasso_ranked)} '
      f'non-zero, top {len(top5_lasso)} by |coefficient|:')
for c, w in lasso_ranked[:TOP_N]:
    print(f'  {c}: {w:.4f}')

Lasso (alpha=0.0153) nominated 12 non-zero, top 5 by |coefficient|:
  us_credit_qoq_growth_L6: 0.7711
  us_house_price_yoy_L3: 0.5517
  us_consumer_confidence_L2: 0.3635
  us_gdp_yoy_growth_L2: 0.2823
  us_unemployment_L0: 0.2449


In [8]:
# 2.4 - method 3 Random Forest permutation importance
rf_sel = RandomForestRegressor(n_estimators=200, max_depth=4,
                               random_state=SEED, n_jobs=-1)
rf_sel.fit(X_sel_s, y_sel)

perm    = permutation_importance(rf_sel, X_sel_s, y_sel, n_repeats=20,
                                 random_state=SEED, scoring='neg_mean_absolute_error')
rf_imp  = dict(zip(CANDIDATES, perm.importances_mean))
top5_rf = sorted(rf_imp, key=rf_imp.get, reverse=True)[:TOP_N]

print(f'Random Forest permutation importance top {TOP_N}:')
for v in top5_rf:
    print(f'  {v}: {rf_imp[v]:.4f}')

Random Forest permutation importance top 5:
  us_credit_qoq_growth_L6: 0.5887
  us_unemployment_L0: 0.1277
  us_house_price_yoy_L3: 0.1194
  us_indprod_yoy_L3: 0.0616
  us_cpi_L0: 0.0468


In [9]:
# 2.5 - method 4 Gradient Boosting importance
gb_sel = GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                   learning_rate=0.05, random_state=SEED)
gb_sel.fit(X_sel_s, y_sel)

gb_imp  = dict(zip(CANDIDATES, gb_sel.feature_importances_))
top5_gb = sorted(gb_imp, key=gb_imp.get, reverse=True)[:TOP_N]

print(f'Gradient Boosting importance top {TOP_N}:')
for v in top5_gb:
    print(f'  {v}: {gb_imp[v]:.4f}')

Gradient Boosting importance top 5:
  us_credit_qoq_growth_L6: 0.4076
  us_house_price_yoy_L3: 0.1453
  us_unemployment_L0: 0.1306
  us_indprod_yoy_L3: 0.1032
  us_sp500_log_ret_L4: 0.0673


In [10]:
# 2.6 - Table 2.1 consensus vote
nominations = {'AIC': top5_aic, 'Lasso': top5_lasso, 'RF': top5_rf, 'GB': top5_gb}

# Rank within each method's top-N; variables not nominated get TOP_N + 1.
vote_counts, mean_ranks = {}, {}
for c in CANDIDATES:
    ranks = [(m.index(c) + 1) if c in m else TOP_N + 1 for m in nominations.values()]
    vote_counts[c] = sum(1 for m in nominations.values() if c in m)
    mean_ranks[c]  = np.mean(ranks)

vote_df = pd.DataFrame({
    'Variable':  CANDIDATES,
    'Votes':     [vote_counts[c] for c in CANDIDATES],
    'Mean rank': [round(mean_ranks[c], 2) for c in CANDIDATES],
    **{m: ['Yes' if c in lst else 'No' for c in CANDIDATES]
       for m, lst in nominations.items()},
}).sort_values(['Votes', 'Mean rank'], ascending=[False, True]).reset_index(drop=True)

print('Table 2.1 - Four-method consensus vote')
display(vote_df.set_index('Variable'))

consensus = vote_df.loc[vote_df['Votes'] >= VOTE_THRESHOLD, 'Variable'].tolist()

# Pre-specified tie-break: if the vote returns fewer than 4 or more than 8, fall back to
# the top 6 by vote count then mean rank. Recorded whether or not it fires.
if len(consensus) < 4 or len(consensus) > 8:
    consensus = vote_df['Variable'].head(6).tolist()
    print('\nTie-break applied: vote returned an out-of-range set, top 6 taken.')
else:
    print(f'\nTie-break not triggered ({len(consensus)} variables, within 4-8).')

FEATURES_EQ511         = consensus + ['covid_dummy']
FEATURES_EQ511_NODUMMY = consensus

unval = [c for c in consensus if any(c.startswith(u) for u in UNVALIDATED_STAGE_A)]

print(f'\nRetained ({len(consensus)} of {len(CANDIDATES)}):')
for c in consensus:
    print(f'  {c}{"  [Stage A unvalidated]" if c in unval else ""}')
print(f'\nRejected: {[c for c in CANDIDATES if c not in consensus]}')
print(f'\nThis set defines Equation 5.11. The equation is a result, reported in '
      f'Section 6, not a methodology pre-specification.')
print(f'{len(unval)} of {len(consensus)} retained variables lack Stage A DM validation.')

Table 2.1 - Four-method consensus vote


,Votes,Mean rank,AIC,Lasso,RF,GB
Variable,,,,,,
us_credit_qoq_growth_L6,4,2.00,Yes,Yes,Yes,Yes
us_unemployment_L0,4,3.00,Yes,Yes,Yes,Yes
us_house_price_yoy_L3,3,3.25,No,Yes,Yes,Yes
us_gdp_yoy_growth_L2,2,4.25,Yes,Yes,No,No
us_consumer_confidence_L2,2,4.75,Yes,Yes,No,No
us_cpi_L0,2,5.00,Yes,No,Yes,No
us_indprod_yoy_L3,2,5.00,No,No,Yes,Yes
us_sp500_log_ret_L4,1,5.75,No,No,No,Yes
us_bond_yield_10y_d1_L2,0,6.00,No,No,No,No



Tie-break not triggered (7 variables, within 4-8).

Retained (7 of 12):
  us_credit_qoq_growth_L6  [Stage A unvalidated]
  us_unemployment_L0
  us_house_price_yoy_L3
  us_gdp_yoy_growth_L2
  us_consumer_confidence_L2  [Stage A unvalidated]
  us_cpi_L0
  us_indprod_yoy_L3  [Stage A unvalidated]

Rejected: ['us_bond_yield_10y_d1_L2', 'us_sp500_log_ret_L4', 'us_vix_log_ret_L0', 'us_oil_yoy_L2', 'us_reer_diff_L0']

This set defines Equation 5.11. The equation is a result, reported in Section 6, not a methodology pre-specification.
3 of 7 retained variables lack Stage A DM validation.


In [11]:
# 2.7 - sensitivity REER excluded
def run_vote(candidates, df_block, top_n=TOP_N, threshold=VOTE_THRESHOLD):
    """Full four-method vote on an arbitrary candidate list. Used for sensitivity checks;
    the primary vote above is run cell by cell so each method's output is visible."""
    s  = df_block[candidates + [TARGET]].dropna()
    Xd = s[candidates]
    y  = s[TARGET].values
    Xs = StandardScaler().fit_transform(Xd)

    a = backward_stepwise_aic(Xd, y, candidates, top_n)

    lc = LassoCV(cv=TimeSeriesSplit(n_splits=5), max_iter=10000,
                 random_state=SEED).fit(Xs, y)
    l  = [c for c, _ in sorted([(c, abs(w)) for c, w in zip(candidates, lc.coef_)
                                if abs(w) > 1e-6],
                               key=lambda t: t[1], reverse=True)[:top_n]]

    rf = RandomForestRegressor(n_estimators=200, max_depth=4,
                               random_state=SEED, n_jobs=-1).fit(Xs, y)
    pi = permutation_importance(rf, Xs, y, n_repeats=20, random_state=SEED,
                                scoring='neg_mean_absolute_error')
    r  = [candidates[i] for i in np.argsort(pi.importances_mean)[::-1][:top_n]]

    gb = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05,
                                   random_state=SEED).fit(Xs, y)
    g  = [candidates[i] for i in np.argsort(gb.feature_importances_)[::-1][:top_n]]

    counts = {c: sum(c in lst for lst in [a, l, r, g]) for c in candidates}
    return sorted([c for c in candidates if counts[c] >= threshold],
                  key=lambda c: -counts[c]), len(s)

# Guard: run_vote reimplements the four methods, so confirm it reproduces the explicit
# vote above before any sensitivity result is trusted.
_check, _ = run_vote(CANDIDATES, df_train)
assert set(_check) == set(consensus), 'run_vote diverges from the explicit primary vote'

# REER truncates the selection sample by 13 quarters (no data before 1994 Q2). The
# justification for re-running without it is data availability, which is knowable without
# reference to any vote outcome, so this is not result-dependent selection.
cand_no_reer     = [c for c in CANDIDATES if not c.startswith('us_reer')]
sens_set, sens_n = run_vote(cand_no_reer, df_train)

print(f'Primary vote : {len(train_sel)} quarters, {len(consensus)} retained')
print(f'Without REER : {sens_n} quarters, {len(sens_set)} retained')
print(f'\nSensitivity set: {sens_set}')

added   = [c for c in sens_set if c not in consensus]
dropped = [c for c in consensus if c not in sens_set]
if not added and not dropped:
    print('\nIdentical to the primary vote. REER truncation does not affect selection.')
else:
    print(f'\nDiffers from primary vote. Added: {added} | Dropped: {dropped}')
    print('This difference must be reported in Section 6.')

Primary vote : 107 quarters, 7 retained
Without REER : 117 quarters, 7 retained

Sensitivity set: ['us_gdp_yoy_growth_L2', 'us_credit_qoq_growth_L6', 'us_unemployment_L0', 'us_house_price_yoy_L3', 'us_cpi_L0', 'us_consumer_confidence_L2', 'us_indprod_yoy_L3']

Identical to the primary vote. REER truncation does not affect selection.


In [12]:
# 2.8 - sensitivity nomination width
# TOP_N = 5 is the pre-registered value. This records how the retained set would change
# under alternatives. Reported as sensitivity; it does not revise the primary set.
print(f'{"top_n":>6} {"Kept":>5}  Variables')
for tn in [4, 5, 6, 7]:
    s, _ = run_vote(CANDIDATES, df_train, top_n=tn)
    mark = '  <- pre-registered' if tn == TOP_N else ''
    print(f'{tn:>6} {len(s):>5}  {[c.replace("us_", "") for c in s]}{mark}')

 top_n  Kept  Variables
     4     6  ['unemployment_L0', 'credit_qoq_growth_L6', 'house_price_yoy_L3', 'gdp_yoy_growth_L2', 'consumer_confidence_L2', 'indprod_yoy_L3']
     5     7  ['unemployment_L0', 'credit_qoq_growth_L6', 'house_price_yoy_L3', 'gdp_yoy_growth_L2', 'cpi_L0', 'consumer_confidence_L2', 'indprod_yoy_L3']  <- pre-registered
     6     8  ['unemployment_L0', 'credit_qoq_growth_L6', 'house_price_yoy_L3', 'cpi_L0', 'gdp_yoy_growth_L2', 'consumer_confidence_L2', 'sp500_log_ret_L4', 'indprod_yoy_L3']
     7     8  ['gdp_yoy_growth_L2', 'unemployment_L0', 'cpi_L0', 'credit_qoq_growth_L6', 'house_price_yoy_L3', 'indprod_yoy_L3', 'consumer_confidence_L2', 'sp500_log_ret_L4']


### Expected signs of the coefficients, and reasoning

Setting the expected sign of each coefficient before fitting is what makes the sign check
meaningful. Two variables are recorded as ambiguous because theory points both ways,
and marking them as such is more honest than picking a direction retrospectively.

The prior is the direction of the effect on the delinquency rate.

| Variable | Lag | Prior | Economic channel |
|---|---|---|---|
| GDP year-on-year growth | 2 | negative | Output growth raises incomes and employment, so borrowers service debt more easily. The two-quarter lag reflects the delay between a slowdown and missed payments. |
| Unemployment rate | 0 | positive | Job loss directly removes the income a borrower needs to pay. Credit card defaults can follow within weeks, unlike mortgages, so the effect is contemporaneous. |
| CPI inflation | 0 | ambiguous | Inflation erodes the real value of existing nominal debt, which helps borrowers. It also erodes real income and raises the cost of essentials, which hurts them. Which dominates depends on whether wages keep pace. |
| Consumer confidence | 2 | negative | Confidence proxies for expected future income. Households expecting stability borrow and repay more reliably. It also leads spending decisions, hence the two-quarter lag. |
| 10-year bond yield, first difference | 2 | positive | Rising rates raise debt service costs on variable-rate borrowing and tighten credit supply. Enters as a change rather than a level because the level was non-stationary. |
| Credit quarter-on-quarter growth | 6 | positive | Rapid credit expansion means rising household leverage and, typically, lending to progressively weaker borrowers. Delinquency follows with a long delay as loans season and income shocks arrive. Six quarters is the longest lag in the candidate set and is consistent with the credit cycle literature. |
| S&P 500 log return | 4 | negative | Equity gains raise household wealth and support balance sheets. The channel is indirect, working through the share of households holding financial assets, hence the weak expected effect. |
| VIX log return | 0 | positive | Volatility signals financial stress and tightening conditions. Contemporaneous because markets react before the real economy. |
| House price year-on-year growth | 3 | negative | Housing is the largest household asset and the main source of collateral. Rising prices allow refinancing and give borrowers an incentive to protect their credit record. |
| Industrial production year-on-year | 3 | negative | A second measure of real activity, weighted toward manufacturing employment. |
| Oil price year-on-year | 2 | positive | Energy is a non-discretionary expense. Rising costs squeeze the residual income available for debt service, with the strongest effect on lower-income borrowers. |
| Real effective exchange rate, first difference | 0 | ambiguous | Appreciation reduces import prices, helping consumers, but hurts exporters and manufacturing employment. The net effect on household credit is indirect and the sign is not predictable a priori. |

### How to read the sign check

A coefficient matching its prior supports the interpretation that the model has found the
economic channel rather than a statistical artefact. A coefficient contradicting a strong
prior needs explanation before the model can be presented as interpretable.

The most common benign explanation is collinearity. When several variables measure the same
underlying channel, the regression assigns them coefficients that jointly fit the data but
individually mean little. A variable can take the opposite sign to its true marginal effect
because it is absorbing variance the others cannot. This is a suppression effect, and the
signature is a coefficient that changes sign as correlated variables are added and becomes
statistically significant only in the full specification.

The candidate set contains three measures of real activity: GDP growth, industrial
production and unemployment. Collinearity among them is expected, which is why the final
diagnostic cell traces one coefficient across nested specifications rather than reporting the
full-model estimate alone.

Sign stability between the 12-variable and 7-variable fits is reported for the same reason.
A coefficient that keeps its sign when five variables are removed is more trustworthy than
one that flips.

In [13]:
# 2.9 - common diagnostic sample
# All diagnostic fits are restricted to a single sample so that coefficient comparisons
# reflect specification changes only, not sample composition. Using the all-12 complete
# case index means every nested fit below sees identical quarters.
COMMON_IDX = df_train[CANDIDATES + [TARGET, 'covid_dummy']].dropna().index
print(f'Common diagnostic sample: {len(COMMON_IDX)} quarters '
      f'({q(COMMON_IDX.min())} to {q(COMMON_IDX.max())})')

Common diagnostic sample: 107 quarters (1994 Q2 to 2020 Q4)


In [14]:
# 2.10 - Table 2.2 joint sign diagnostic
# Economic priors set before fitting. Ambiguous cases are recorded as such rather than
# assigned a direction, so the check cannot be passed by hindsight.
SIGN_PRIORS = {
    'us_gdp_yoy_growth_L2':      ('-', 'growth reduces defaults'),
    'us_unemployment_L0':        ('+', 'joblessness raises defaults'),
    'us_cpi_L0':                 ('?', 'erodes real debt burden but also real income'),
    'us_consumer_confidence_L2': ('-', 'confidence reduces defaults'),
    'us_bond_yield_10y_d1_L2':   ('+', 'rising rates raise debt service'),
    'us_credit_qoq_growth_L6':   ('+', 'credit expansion precedes over-leverage'),
    'us_sp500_log_ret_L4':       ('-', 'equity gains support household balance sheets'),
    'us_vix_log_ret_L0':         ('+', 'volatility signals stress'),
    'us_house_price_yoy_L3':     ('-', 'housing wealth and collateral support repayment'),
    'us_indprod_yoy_L3':         ('-', 'activity reduces defaults'),
    'us_oil_yoy_L2':             ('+', 'energy costs squeeze disposable income'),
    'us_reer_diff_L0':           ('?', 'competitiveness channel is indirect'),
}

def fit_hac(features, index=COMMON_IDX):
    s = df_train.loc[index, features + [TARGET]]
    X = sm.add_constant(s[features].values, has_constant='add')
    return sm.OLS(s[TARGET].values, X).fit(cov_type='HAC', cov_kwds={'maxlags': 4})

ols_all12 = fit_hac(CANDIDATES + ['covid_dummy'])
ols_sel   = fit_hac(FEATURES_EQ511)
sel_coefs = dict(zip(FEATURES_EQ511, ols_sel.params[1:]))

rows = []
for i, c in enumerate(CANDIDATES):
    b12   = ols_all12.params[i + 1]
    prior = SIGN_PRIORS[c][0]
    got   = '+' if b12 > 0 else '-'
    b7    = sel_coefs.get(c, np.nan)
    rows.append({
        'Variable':      c,
        'Prior':         prior,
        'All 12':        round(b12, 4),
        'p':             round(ols_all12.pvalues[i + 1], 4),
        'Selected 7':    round(b7, 4) if c in sel_coefs else '-',
        'Matches prior': 'n/a' if prior == '?' else ('Yes' if got == prior else 'NO'),
        'Sign stable':   ('-' if c not in sel_coefs
                          else 'Yes' if np.sign(b12) == np.sign(b7) else 'FLIP'),
    })

print(f'Table 2.2 - Joint sign diagnostic (Newey-West HAC, maxlags=4, n={len(COMMON_IDX)}). '
      f'Diagnostic only.')
display(pd.DataFrame(rows).set_index('Variable'))

print(f'Joint 12-variable fit: R2={ols_all12.rsquared:.3f} | '
      f'DW={durbin_watson(ols_all12.resid):.3f} | AIC={ols_all12.aic:.1f}')


viol = [r['Variable'] for r in rows if r['Matches prior'] == 'NO']
flip = [r['Variable'] for r in rows if r['Sign stable'] == 'FLIP']
print(f'Prior violations: {viol if viol else "none"}')
print(f'Sign flips between the 12-variable and 7-variable fits: {flip if flip else "none"}')

Table 2.2 - Joint sign diagnostic (Newey-West HAC, maxlags=4, n=107). Diagnostic only.


,Prior,All 12,p,Selected 7,Matches prior,Sign stable
Variable,,,,,,
us_gdp_yoy_growth_L2,-,0.1769,0.0133,0.1771,NO,Yes
us_unemployment_L0,+,0.2499,0.0008,0.2486,Yes,Yes
us_cpi_L0,?,0.1104,0.1116,0.1625,n/a,Yes
us_consumer_confidence_L2,-,-0.3492,0.0000,-0.3173,Yes,Yes
us_bond_yield_10y_d1_L2,+,0.0401,0.6957,-,Yes,-
us_credit_qoq_growth_L6,+,0.9238,0.0000,0.9175,Yes,Yes
us_sp500_log_ret_L4,-,-0.0356,0.9734,-,Yes,-
us_vix_log_ret_L0,+,0.4638,0.0221,-,Yes,-
us_house_price_yoy_L3,-,-0.0956,0.0000,-0.0973,Yes,Yes


Joint 12-variable fit: R2=0.729 | DW=0.956 | AIC=219.1
Prior violations: ['us_gdp_yoy_growth_L2']
Sign flips between the 12-variable and 7-variable fits: none


In [15]:
# 2.11 - Table 2.3 multicollinearity
# Reported on the selected model's own sample, not COMMON_IDX: variance inflation should
# describe the model actually fitted, so n here exceeds the diagnostic sample above.
vif_data = df_train[FEATURES_EQ511].dropna()
vif_mat  = sm.add_constant(vif_data.values, has_constant='add')

vif_df = pd.DataFrame({
    'Variable': list(vif_data.columns),
    'VIF': [round(variance_inflation_factor(vif_mat, i + 1), 2)
            for i in range(vif_data.shape[1])],
}).sort_values('VIF', ascending=False).reset_index(drop=True)

# Thresholds from Bellotti and Crook (2012), Section 3.2: below 5 for OLS, below 10 for
# regularised and nonlinear models, which tolerate correlated inputs.
vif_df['Assessment'] = ['OK for OLS' if v < 5 else
                        'OK for regularised only' if v < 10 else 'High'
                        for v in vif_df['VIF']]

print(f'Table 2.3 - Variance Inflation Factors (n={len(vif_data)})')
display(vif_df.set_index('Variable'))

high = vif_df.loc[vif_df['VIF'] >= 5, 'Variable'].tolist()
print(f'Above 5 (OLS threshold): {high if high else "none"}')

Table 2.3 - Variance Inflation Factors (n=117)


,VIF,Assessment
Variable,,
us_gdp_yoy_growth_L2,3.65,OK for OLS
us_unemployment_L0,3.63,OK for OLS
us_indprod_yoy_L3,3.04,OK for OLS
us_house_price_yoy_L3,2.51,OK for OLS
us_consumer_confidence_L2,2.41,OK for OLS
us_credit_qoq_growth_L6,1.65,OK for OLS
us_cpi_L0,1.28,OK for OLS
covid_dummy,1.26,OK for OLS


Above 5 (OLS threshold): none


In [16]:
# 2.12 - GDP sign trace
# GDP enters positive against a negative prior, significantly. This traces where the
# reversal happens: alone, then conditioned on the other activity measures. All four fits
# use COMMON_IDX so the coefficient movement is attributable to conditioning alone.
gdp, unemp, indprod = 'us_gdp_yoy_growth_L2', 'us_unemployment_L0', 'us_indprod_yoy_L3'

for label, feats in [('GDP alone', [gdp]),
                     ('GDP + unemployment', [gdp, unemp]),
                     ('GDP + unemp + indprod', [gdp, unemp, indprod]),
                     ('Full selected set', FEATURES_EQ511)]:
    sub = df_train.loc[COMMON_IDX, feats + [TARGET]]
    m = sm.OLS(sub[TARGET].values,
               sm.add_constant(sub[feats].values, has_constant='add')
               ).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
    k = feats.index(gdp) + 1          # +1 for the intercept
    print(f'{label:<24} GDP coef = {m.params[k]:+.4f}  '
          f'(p={m.pvalues[k]:.4f}, n={len(sub)})')

s       = df_train.loc[COMMON_IDX, [gdp, TARGET]]
vif_gdp = vif_df.loc[vif_df['Variable'] == gdp, 'VIF'].iloc[0]
print(f'\nPearson correlation, GDP vs delinquency: {s[gdp].corr(s[TARGET]):+.4f}')
print(f'GDP variance explained by other selected regressors: {1 - 1/vif_gdp:.1%}')

GDP alone                GDP coef = -0.0824  (p=0.5836, n=107)
GDP + unemployment       GDP coef = -0.0255  (p=0.8350, n=107)
GDP + unemp + indprod    GDP coef = +0.0513  (p=0.6335, n=107)
Full selected set        GDP coef = +0.1771  (p=0.0148, n=107)

Pearson correlation, GDP vs delinquency: -0.1371
GDP variance explained by other selected regressors: 72.6%


## Section 3 - Feature Matrix Construction

### Why fit four separate versions of the model?

Three methodological choices in this project are either contested in the literature or rest
on judgement and suggestions of our supervisors and industry partners rather than a direct citation. Running alternative specifications that each
change exactly one choice makes it possible to see how much each choice drives the final
forecast.

This is standard applied practice and it is also what the IFRS 9 framework suggests:
institutions must be able to document that their Expected Credit Loss estimates do not
depend critically on arbitrary modelling decisions.

### The four runs

| Run | Target | Features | COVID dummy | Training quarters | What it tests |
|---|---|---|---|---|---|
| Primary | Spline-adjusted | Vote-selected set | On | 117 | Main result |
| R1 | Spline-adjusted | Vote-selected set | Off | 117 | Is the dummy doing the work the macro variables should do? |
| R2 | Spline-adjusted | All 13 candidate columns | On | 107 | Does excluding five variables cost accuracy? Does the CPI lag choice matter? |
| R3 | Raw observed, COVID quarters removed | Vote-selected set | Not applicable | 113 | Does the spline reconstruction matter, or would deletion give the same answer? |

### Primary

The vote-selected regressors plus the COVID dummy, fitted on the spline-adjusted target
across the full training block. Every claim in Section 5 and the production forecast in
Section 6 come from this specification.

### R1, testing the COVID dummy

The dummy is removed and nothing else changes. The question is whether the macroeconomic
variables carry the signal on their own, or whether the dummy is absorbing variation they
should be explaining.

There is a specific reason to worry here. The dummy takes the value one in 2020 Q1 to 2022 Q2,
but the training block ends in 2020 Q4, so the dummy is identified from only four training
observations. A coefficient estimated from four data points is imprecise regardless of its
p-value. If R1 stays close to the primary run, the dummy is a modest addition and the macro
variables are sufficient. If R1 diverges, the dummy is carrying substantial weight and the
forecast is sensitive to how the COVID period is treated.

Bellotti and Crook (2012) argue against time dummies in credit risk models on the grounds
that they absorb genuine macroeconomic signal rather than purely structural breaks. R1 tests
their concern directly.

### R2, testing the vote

All 13 available candidate columns enter with no vote applied, and a Lasso reduction step is
retained in case the penalty binds. The question is whether the consensus vote's exclusion of
five variables cost anything, and whether the CPI lag choice matters given that CPI enters
the primary specification at lag 0 while lag 6 was also identified in the EDA.

Two points about this run - first, it is an unrestricted specification rather than
a genuinely lag-rich one. The Stage A output holds one lag per variable plus the second CPI
lag, so multiple lags per variable are not available to test. A true lag-richness test would
need columns that do not exist in the input file. Second, the Lasso penalty does not bind on
this data, so no reduction actually occurs and R2 is the full 13-column set.

R2 also changes the sample. Reintroducing the exchange rate variable pulls the start date
forward to 1994 Q2 and costs 10 training quarters. A supplementary matrix therefore fits the
primary specification on R2's own sample, so that the comparison in Section 6 isolates the
change in specification from the change in sample composition. Without that control, a worse
R2 result could be attributed either to the extra variables or to the shorter sample, and the
two could not be separated.

### R3, testing the spline

The raw observed target replaces the reconstructed one, and the COVID quarters are removed
from the training data rather than reconstructed. This is the approach Liu et al. (2025) take.

The dummy is dropped in this run because with the COVID quarters gone it would be constant
zero across the training sample and the design matrix would be singular.

R3 is a narrower test than the ten-quarter distortion window suggests. Only four of those
quarters fall inside the training block, since the block ends in 2020 Q4 (BECAUSE THAT IS WHEN OUR TRAINING PERIOD ENDS); the remaining six
lie inside the evaluation window. So R3 removes four training quarters, not ten.

One property makes R3 directly comparable to the primary run despite the different target.
The spline modifies only 2020 Q1 to 2022 Q2, so from 2022 Q3 onward the adjusted and raw
series coincide. On the clean sub-window, which is the primary reporting window, both runs are
scored against identical values. Any difference in their accuracy is therefore attributable
to the training treatment alone rather than to being graded against different numbers.

### How to judge whether a run has diverged

A run counts as materially divergent if its forecast departs from the primary path by more
than the primary model's own clean-window error. A gap smaller than the model's own error is
inside its tolerance and should not be read as a meaningful difference.

That threshold is pre-registered but lenient, so Section 6 also expresses each gap as a
percentage of the forecast level. A gap can sit inside the formal threshold and still be
economically substantial, and reporting both keeps the pre-registered verdict while making
the magnitude visible.

In [17]:
# 3.1 - matrix builder
def build_matrices(features, target, blocks=None):
    """Aligned X, y arrays for every block a run needs.

    Complete cases only: a quarter is dropped if the target or any feature is missing.
    Returns a dict keyed by block name holding (X, y, index), except 'fcst' which holds
    (X, index) since no target exists beyond 2025 Q4. Pass `blocks` to override any
    default block, which is how R3 substitutes its COVID-excluded training data.
    """
    B = {'train': df_train, 'tune': df_tune, 'val': df_val,
         'eval': df_eval, 'full': df_full, 'fcst': df_fcst}
    if blocks:
        B.update(blocks)

    def prep(block):
        s = block[features + [target]].dropna()
        return s[features].values, s[target].values, s.index

    m = {k: prep(B[k]) for k in ['train', 'tune', 'val', 'eval', 'full']}
    s_fc = B['fcst'][features].dropna()
    m['fcst'] = (s_fc.values, s_fc.index)

    # Evaluation and forecast blocks must be complete. A dropped row in either would
    # silently shorten the horizon or misalign forecasts against benchmarks in Section 5.
    assert len(m['eval'][2]) == len(B['eval']), \
        f'eval incomplete: {len(m["eval"][2])} of {len(B["eval"])}'
    assert len(m['fcst'][1]) == len(B['fcst']), \
        f'forecast incomplete: {len(m["fcst"][1])} of {len(B["fcst"])}'
    return m

print('build_matrices defined.')

build_matrices defined.


In [18]:
# 3.2 - primary run and R1
mats_primary = build_matrices(FEATURES_EQ511, TARGET)
mats_r1      = build_matrices(FEATURES_EQ511_NODUMMY, TARGET)

for name, m, feats in [('Primary', mats_primary, FEATURES_EQ511),
                       ('R1', mats_r1, FEATURES_EQ511_NODUMMY)]:
    print(f'{name:<8} {len(feats)} features | train {len(m["train"][1])} | '
          f'full {len(m["full"][1])} | eval {len(m["eval"][1])} | fcst {len(m["fcst"][1])}')

Primary  8 features | train 117 | full 137 | eval 20 | fcst 20
R1       7 features | train 117 | full 137 | eval 20 | fcst 20


In [19]:
# 3.3 - R2 unrestricted specification
# All 13 available candidate columns, no vote. The Stage A output holds one lag per
# variable plus a second CPI lag, so this tests whether the vote's exclusion of five
# variables costs accuracy, and whether CPI's lag choice matters.
r2_train = df_train[CANDIDATES_R2 + [TARGET]].dropna()
X_r2     = StandardScaler().fit_transform(r2_train[CANDIDATES_R2])
y_r2     = r2_train[TARGET].values

lasso_r2 = LassoCV(cv=TimeSeriesSplit(n_splits=5), max_iter=10000,
                   random_state=SEED).fit(X_r2, y_r2)
kept_r2  = [c for c, w in zip(CANDIDATES_R2, lasso_r2.coef_) if abs(w) > 1e-6]

print(f'R2 Lasso alpha={lasso_r2.alpha_:.4f} | '
      f'retained {len(kept_r2)} of {len(CANDIDATES_R2)}')
if len(kept_r2) == len(CANDIDATES_R2):
    print('Penalty does not bind, so no reduction occurs. R2 is the full unrestricted set.')
else:
    print(f'Dropped: {[c for c in CANDIDATES_R2 if c not in kept_r2]}')

FEATURES_SPECD = kept_r2 + ['covid_dummy']
mats_r2        = build_matrices(FEATURES_SPECD, TARGET)

print(f'\nR2       {len(FEATURES_SPECD)} features | train {len(mats_r2["train"][1])} | '
      f'full {len(mats_r2["full"][1])}')

R2 Lasso alpha=0.0153 | retained 13 of 13
Penalty does not bind, so no reduction occurs. R2 is the full unrestricted set.

R2       14 features | train 107 | full 127


In [20]:
# 3.4 - R3 raw target, COVID quarters excluded
def drop_covid(block):
    return block[~((block.index >= COVID_START) & (block.index <= COVID_END))]

# covid_dummy is removed rather than left in: with the COVID quarters gone it would be
# constant zero across training and the design matrix singular.
FEATURES_R3 = [f for f in FEATURES_EQ511 if f != 'covid_dummy']

mats_r3 = build_matrices(FEATURES_R3, TARGET_RAW, blocks={
    'train': drop_covid(df_train),
    'tune':  drop_covid(df_tune),
    'val':   drop_covid(df_val),
    'full':  drop_covid(df_full),
})

# The spline only modifies 2020 Q1 to 2022 Q2, so on the clean sub-window the adjusted and
# raw targets coincide. R3 and the primary run are therefore scored against identical
# values there, and any difference between them is due to training treatment alone.
clean_same = np.allclose(df.loc[CLEAN_START:CLEAN_END, TARGET],
                         df.loc[CLEAN_START:CLEAN_END, TARGET_RAW])

print(f'R3       {len(FEATURES_R3)} features | train {len(mats_r3["train"][1])} | '
      f'full {len(mats_r3["full"][1])} (COVID quarters excluded)')
print(f'Training quarters removed: {len(df_train) - len(drop_covid(df_train))}')
print(f'Spline and raw targets identical on the clean sub-window: {clean_same}')

R3       7 features | train 113 | full 127 (COVID quarters excluded)
Training quarters removed: 4
Spline and raw targets identical on the clean sub-window: True


In [21]:
# 3.5 - Table 3.1 run register
RUNS = {
    'Primary': dict(mats=mats_primary, features=FEATURES_EQ511,        target=TARGET,
                    dummy='On',  tests='main result'),
    'R1':      dict(mats=mats_r1,      features=FEATURES_EQ511_NODUMMY, target=TARGET,
                    dummy='Off', tests='is the COVID dummy driving results'),
    'R2':      dict(mats=mats_r2,      features=FEATURES_SPECD,         target=TARGET,
                    dummy='On',  tests='does excluding five variables cost accuracy'),
    'R3':      dict(mats=mats_r3,      features=FEATURES_R3,            target=TARGET_RAW,
                    dummy='n/a', tests='does the spline adjustment matter'),
}

run_df = pd.DataFrame([
    {'Run': k,
     'Target': 'spline' if v['target'] == TARGET else 'raw',
     'Features': len(v['features']),
     'Dummy': v['dummy'],
     'Train n': len(v['mats']['train'][1]),
     'Full n': len(v['mats']['full'][1]),
     'What it tests': v['tests']}
    for k, v in RUNS.items()
])

print('Table 3.1 - Robustness run register')
display(run_df.set_index('Run'))

Table 3.1 - Robustness run register


,Target,Features,Dummy,Train n,Full n,What it tests
Run,,,,,,
Primary,spline,8,On,117,137,main result
R1,spline,7,Off,117,137,is the COVID dummy driving results
R2,spline,14,On,107,127,does excluding five variables cost accuracy
R3,raw,7,n/a,113,127,does the spline adjustment matter


In [22]:
# 3.6 - sample overlap between runs
# R2 reintroduces REER and so starts at 1994 Q2, giving it 10 fewer training quarters than
# the primary run. A supplementary matrix fits the primary specification on R2's sample so
# that the R2 comparison in Section 5 isolates specification from sample.
mats_primary_r2sample = build_matrices(
    FEATURES_EQ511, TARGET,
    blocks={'train': df_train.loc[mats_r2['train'][2]],
            'tune':  df_tune.loc[df_tune.index.intersection(mats_r2['train'][2])],
            'val':   df_val.loc[df_val.index.intersection(mats_r2['train'][2])],
            'full':  df_full.loc[mats_r2['full'][2]]})

dummy_n = int(df_train.loc[mats_primary['train'][2], 'covid_dummy'].sum())
print(f'covid_dummy = 1 on {dummy_n} of {len(mats_primary["train"][1])} training quarters')
print(f'Primary on R2 sample: train {len(mats_primary_r2sample["train"][1])} '
      f'(vs {len(mats_primary["train"][1])} on its own sample)')

covid_dummy = 1 on 4 of 117 training quarters
Primary on R2 sample: train 107 (vs 117 on its own sample)


## Section 4 - Model Fitting

Nine models are fitted on the primary specification: OLS, three regularised linear models,
two kernel methods, two tree ensembles and a small neural network. OLS is the pre-registered
primary model and the deliverable. The other eight are challengers.

### The evaluation design: single origin

In a rolling-origin backtest, the model is retrained at each quarter and produces many
overlapping forecasts. That generates more test observations, but it does not match how the
model would actually be used. In production, a company like BDO would likely train on available history and projects 20
quarters forward without retraining at each step. A single origin at 2020 Q4 replicates that.

The cost is that only one 20-quarter error path is observed rather than many, which limits
statistical power. This is the direct reason no comparison in Section 5 reaches significance,
and it is a property of the design rather than a finding about the models.

### Inner tune and validation split inside the training block

Eight of the nine models have hyperparameters, settings such as the regularisation strength
in Ridge or the tree depth in Random Forest, that cannot be estimated from the data in a
single pass. They have to be searched over a grid of candidate values.

Searching that grid on the full training block and then evaluating on the same block would
select whichever setting fitted the training data best, which is not the same as the setting
that generalises. The inner split solves this: the grid is searched by fitting on 1991 Q1 to
2016 Q4 and scoring on 2017 Q1 to 2020 Q4, and the winning configuration is then refitted on
the whole training block. Neither step ever touches the evaluation window.

### A caution about the validation window

Delinquency was flat and historically low across 2017 to 2020, so a model that predicts close
to a constant scores well there. Validation error therefore partly measures how little a model
moves rather than how well it forecasts, and it does not predict performance on 2021 to 2025.

The neural network is the clearest case. Its validation error is lower than its training
error, which normally cannot happen. The cause is that a large L2 penalty shrinks it toward a
near-flat prediction that suits a flat validation period while missing the 2009 peak
entirely. This is worth knowing before reading the validation numbers as a ranking.

### Why not k-fold cross-validation?

Standard k-fold divides data into random folds, which for time series means some folds train
on future data to predict the past, an advantage the model will not have in deployment.

Bergmeir, Hyndman and Koo (2018) show formally that k-fold is valid only for purely
autoregressive models with uncorrelated errors. Figure 4.2 reports a Durbin-Watson statistic
close to 0.94 and a Ljung-Box p-value below 0.0001, so the residuals here are strongly
serially correlated. k-fold would produce misleadingly optimistic validation scores. Rolling
and expanding splits are used throughout instead.

### Grid boundaries

A hyperparameter that wins at the edge of its grid is a sign the search space was too narrow,
because the true optimum may lie outside it. The fitter therefore prints a warning and the
grid is widened and re-run.

Ridge, Lasso and Elastic Net all select the smallest penalty available and converge on the
OLS solution, with validation errors identical to three decimals. Extending those grids
downward is not informative, since it only converges further onto OLS, so the boundary hit is
reported as a finding rather than treated as a defect. Regularisation buys nothing on this
data, which given 117 observations against eight features is unsurprising and mildly
reassuring about the specification.

Parameters with a hard theoretical bound, such as the Elastic Net mixing ratio and the
XGBoost subsample fraction at 1.0, cannot be widened past that bound and are exempt from the
check.

### Two fits per model

Each model is fitted twice. The training fit, on 1991 to 2020, produces the forecasts scored
in Section 5. The production fit, on all history to 2025 Q4, generates the 2026 to 2030
projection in Section 6. Feature scalers are fitted separately within each, on training data
only, so no information from later periods leaks into earlier ones.

### Metrics defined here

The loss functions are defined in this section rather than Section 5 because Figure 4.1 uses
them.

Root mean squared error is the primary metric. It squares each error before averaging and
then takes the square root, so it is expressed in percentage points of the delinquency rate
but gives proportionally more weight to large errors. RMSE is primary here because it matches
the Stage A metric set, which keeps the two stages comparable.

Mean absolute error averages the absolute errors without squaring. It is reported alongside
RMSE, and comparing the two indicates whether a model's error is evenly spread across
quarters or driven by a few bad ones. A model with low MAE and high RMSE is usually close
most of the time and badly wrong occasionally.

Symmetric mean absolute percentage error expresses error as a percentage of the average of
actual and predicted values. It is scale-free and bounded between 0 and 200 per cent, which
makes it useful for comparison across series of different magnitudes.

Forecast skill measures accuracy relative to a reference forecast rather than in absolute
terms. The formula is one minus the ratio of the model's loss to the reference's loss, so a
skill of positive 0.15 means the model's loss is 15 per cent lower than the reference. Zero
means equal, negative means worse. Two properties matter. The reference has to be
constructible at the forecast origin, otherwise the comparison uses information the forecaster
could not have had. And the loss basis matters - skill computed on RMSE and skill computed on
squared error are different numbers, because one is a ratio of roots and the other a ratio of
squares.

### Figures in this section

Figure 4.1 compares training fit against validation error for all 9 models. A
close training fit paired with a high validation error indicates memorisation rather than
skill. XGBoost is the clearest illustration as it fits the training data almost exactly and
generalises far worse, the expected outcome for a boosted ensemble on 117 observations
(Grinsztajn et al. 2022).

Figure 4.2 shows the OLS residuals over time and against a normal distribution. Residuals run
predominantly negative after 2011, meaning the model over-predicts delinquency in the
post-crisis period. Because the specification contains no lagged dependent variable, it has no
mechanism to correct a persistent level error, and that bias carries into the evaluation
window. The residuals are close to normal but strongly autocorrelated, which is why
Newey-West standard errors are used for inference throughout.

NOTE: FOR CHANGING THE VALIDATION METRIC FOR MAE TO RMSE, CODE WAS COMMENTED OUT AND REPLACED IN 4.1, 4.2 AND 4.5

In [23]:
# 4.1 - OLS fitter
def fit_ols(mats):
    """OLS with Newey-West HAC standard errors, maxlags=4 (one year at quarterly
    frequency). Fits twice: on the training block for evaluation, and on the full history
    for the production forecast. The evaluation fit never sees 2021 onward."""
    X_tr, y_tr, idx_tr = mats['train']
    X_ev, y_ev, idx_ev = mats['eval']
    X_fu, y_fu, _      = mats['full']
    X_fc, idx_fc       = mats['fcst']

    add = lambda X: sm.add_constant(X, has_constant='add')

    model = sm.OLS(y_tr, add(X_tr)).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
    prod  = sm.OLS(y_fu, add(X_fu)).fit(cov_type='HAC', cov_kwds={'maxlags': 4})

    return {'model': model, 'prod_model': prod,
            'train_pred': model.predict(add(X_tr)),
            'eval_pred':  model.predict(add(X_ev)),
            'fcst_pred':  prod.predict(add(X_fc)),
            'idx_train': idx_tr, 'idx_eval': idx_ev, 'idx_fcst': idx_fc,
            'y_train': y_tr, 'y_eval': y_ev,
#           'best_params': {}, 'val_mae': np.nan
            'best_params': {}, 'val_rmse': np.nan, 'val_mae': np.nan}

print('fit_ols defined.')

fit_ols defined.


In [24]:
# 4.2 - sklearn fitter with inner tune/validation split
def fit_sklearn_model(estimator, param_grid, mats, label='', scale=True):
    """Grid search on the inner split, refit, then forecast.

    Hyperparameters are searched by fitting on 1991-2016 and scoring on 2017-2020, so the
    evaluation window is never used to choose a configuration. The winning configuration
    is refit on the whole training block for evaluation, and separately on the full
    history for the production forecast. Scalers are fitted on training data only.
    """
    X_tr, y_tr, idx_tr = mats['train']
    X_tu, y_tu, _      = mats['tune']
    X_va, y_va, _      = mats['val']
    X_ev, y_ev, idx_ev = mats['eval']
    X_fu, y_fu, _      = mats['full']
    X_fc, idx_fc       = mats['fcst']

    sc = StandardScaler().fit(X_tr) if scale else None
    t  = (lambda X: sc.transform(X)) if scale else (lambda X: X)

#    best_mae, best_params = np.inf, {}
#    for params in ParameterGrid(param_grid):
#        est = clone(estimator).set_params(**params)
#        est.fit(t(X_tu), y_tu)
#        m = mean_absolute_error(y_va, est.predict(t(X_va)))
#        if m < best_mae:
#            best_mae, best_params = m, params

    # Selection on validation RMSE, matching the primary reporting metric. MAE is also
    # recorded for the winning configuration so both appear on the same basis in 4.5.
    best_rmse, best_mae, best_params = np.inf, np.nan, {}
    for params in ParameterGrid(param_grid):
        est = clone(estimator).set_params(**params)
        est.fit(t(X_tu), y_tu)
        pred = est.predict(t(X_va))
        r = float(np.sqrt(mean_squared_error(y_va, pred)))
        if r < best_rmse:
            best_rmse, best_params = r, params
            best_mae = float(mean_absolute_error(y_va, pred))

    # A winner on a grid edge means the search space was too narrow. Parameters with a hard
    # theoretical bound cannot be widened past it, so selecting that value is not evidence
    # of a truncated grid.
    HARD_BOUNDS = {'l1_ratio': 1.0, 'subsample': 1.0}
    for key, grid in param_grid.items():
        nums = [v for v in grid if isinstance(v, (int, float))]
        if len(nums) != len(grid) or len(grid) < 2:
            continue
        won = best_params.get(key)
        if won in (min(nums), max(nums)) and won != HARD_BOUNDS.get(key):
            print(f'  BOUNDARY: {label} {key}={won} at grid edge')

    final = clone(estimator).set_params(**best_params).fit(t(X_tr), y_tr)

    sc_p = StandardScaler().fit(X_fu) if scale else None
    t_p  = (lambda X: sc_p.transform(X)) if scale else (lambda X: X)
    prod = clone(estimator).set_params(**best_params).fit(t_p(X_fu), y_fu)

    return {'model': final, 'prod_model': prod, 'scaler': sc,
            'train_pred': final.predict(t(X_tr)),
            'eval_pred':  final.predict(t(X_ev)),
            'fcst_pred':  prod.predict(t_p(X_fc)),
            'idx_train': idx_tr, 'idx_eval': idx_ev, 'idx_fcst': idx_fc,
            'y_train': y_tr, 'y_eval': y_ev,
#            'best_params': best_params, 'val_mae': round(best_mae, 4)}
            'best_params': best_params, 'val_rmse': round(best_rmse, 4),
            'val_mae': round(best_mae, 4)}

print('fit_sklearn_model defined.')

fit_sklearn_model defined.


In [25]:
# 4.3 - loss and skill functions
def mae(y, f):
    """Mean absolute error, in percentage points of the delinquency rate."""
    return mean_absolute_error(y, f)

def rmse(y, f):
    """Root mean squared error. Penalises large errors more heavily than MAE."""
    return np.sqrt(mean_squared_error(y, f))

def smape(y, f):
    """Symmetric mean absolute percentage error. Scale-free, bounded 0-200%."""
    y, f = np.asarray(y), np.asarray(f)
    return 100 * np.mean(np.abs(y - f) / ((np.abs(y) + np.abs(f)) / 2))

def skill(y, f, ref, loss):
    """Forecast skill against a reference forecast: SS = 1 - L(model)/L(reference).

    Positive means the model beats the reference, zero means equal, negative means worse.
    Read as fractional loss reduction: +0.15 is 15% lower loss than the reference. The
    loss basis matters - RMSE-skill and MSE-skill differ, since one is the ratio of roots
    and the other the ratio of squares.
    The reference must be constructible at the forecast origin, so no reference here uses
    information from the evaluation window itself.
    """
    y, f, ref = np.asarray(y), np.asarray(f), np.asarray(ref)
    if loss == 'mae':
        num, den = mae(y, f), mae(y, ref)
    elif loss == 'rmse':
        num, den = rmse(y, f), rmse(y, ref)
    elif loss == 'mse':
        num, den = np.mean((y - f) ** 2), np.mean((y - ref) ** 2)
    else:
        raise ValueError("loss must be 'mae', 'rmse' or 'mse'")
    return np.nan if den <= 0 else 1 - num / den

print('Loss and skill functions defined.')

Loss and skill functions defined.


In [26]:
# 4.4 - hyperparameter grids
# Ridge, Lasso and Elastic Net all select the smallest penalty available and converge on
# the OLS solution. Extending those grids downward is not informative, so they are left
# as-is and the boundary hit is reported as a finding rather than a grid defect.
GRIDS = {
    'Ridge':         (Ridge(),
                      {'alpha': [1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0, 100.0, 1000.0]}),
    'Lasso':         (Lasso(max_iter=50000),
                      {'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.5, 1.0]}),
    'Elastic Net':   (ElasticNet(max_iter=50000),
                      {'alpha': [1e-4, 1e-3, 1e-2, 0.1, 1.0],
                       'l1_ratio': [0.05, 0.2, 0.5, 0.8, 0.95, 1.0]}),
    'KRR':           (KernelRidge(kernel='rbf'),
                      {'alpha': [1e-3, 1e-2, 0.1, 1.0, 10.0],
                       'gamma': [1e-4, 1e-3, 1e-2, 0.1, 1.0]}),
    'SVR':           (SVR(kernel='rbf'),
                      {'C': [0.1, 1.0, 10.0, 100.0, 1000.0],
                       'epsilon': [1e-3, 1e-2, 0.1, 0.5, 1.0, 2.0],
                       'gamma': ['scale']}),
    'Random Forest': (RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
                      {'max_depth': [2, 3, 4, 6, 8, None],
                       'min_samples_leaf': [1, 2, 4, 8, 16]}),
    'XGBoost':       (xgb.XGBRegressor(n_estimators=300, random_state=SEED,
                                       verbosity=0, n_jobs=-1),
                      {'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.3, 0.5],
                       'max_depth': [1, 2, 3, 4, 6],
                       'subsample': [0.6, 0.8, 1.0]}),
    # Single hidden layer, 4-8 units, strong L2, early stopping. Grinsztajn et al. (2022)
    # find trees competitive with deep models at ~10,000 samples; we have 117.
    'MLP':           (MLPRegressor(max_iter=5000, early_stopping=True,
                                   n_iter_no_change=50, random_state=SEED),
                      {'hidden_layer_sizes': [(4,), (6,), (8,)],
                       'alpha': [0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]}),
}

MODEL_NAMES = ['OLS'] + list(GRIDS.keys())
print(f'{len(MODEL_NAMES)} models: {MODEL_NAMES}')
for name, (_, g) in GRIDS.items():
    print(f'  {name:<14} {len(list(ParameterGrid(g))):>3} candidates')

9 models: ['OLS', 'Ridge', 'Lasso', 'Elastic Net', 'KRR', 'SVR', 'Random Forest', 'XGBoost', 'MLP']
  Ridge            8 candidates
  Lasso            8 candidates
  Elastic Net     30 candidates
  KRR             25 candidates
  SVR             30 candidates
  Random Forest   30 candidates
  XGBoost         90 candidates
  MLP             18 candidates


In [27]:
# 4.5 - fit all nine on the primary specification
results = {}

print('Fitting OLS...')
results['OLS'] = fit_ols(mats_primary)
print(f'  R2={results["OLS"]["model"].rsquared:.3f} | '
      f'DW={durbin_watson(results["OLS"]["model"].resid):.3f}')

for name, (est, grid) in GRIDS.items():
    print(f'Fitting {name}...')
    results[name] = fit_sklearn_model(est, grid, mats_primary, label=name)
#    print(f'  {results[name]["best_params"]} | val MAE={results[name]["val_mae"]}')
    print(f'  {results[name]["best_params"]} | val RMSE={results[name]["val_rmse"]} '
          f'| val MAE={results[name]["val_mae"]}')

print(f'\nAll {len(results)} models fitted on the primary specification.')

Fitting OLS...
  R2=0.705 | DW=0.936
Fitting Ridge...
  {'alpha': 1.0} | val RMSE=1.0292 | val MAE=0.7504
Fitting Lasso...
  {'alpha': 0.01} | val RMSE=1.0233 | val MAE=0.7977
Fitting Elastic Net...
  {'alpha': 0.01, 'l1_ratio': 1.0} | val RMSE=1.0233 | val MAE=0.7977
Fitting KRR...
  {'alpha': 0.1, 'gamma': 0.01} | val RMSE=0.7227 | val MAE=0.6419
Fitting SVR...
  {'C': 10.0, 'epsilon': 1.0, 'gamma': 'scale'} | val RMSE=1.2218 | val MAE=0.9725
Fitting Random Forest...
  {'max_depth': 4, 'min_samples_leaf': 8} | val RMSE=1.234 | val MAE=1.0718
Fitting XGBoost...
  {'learning_rate': 0.3, 'max_depth': 2, 'subsample': 1.0} | val RMSE=1.0072 | val MAE=0.7011
Fitting MLP...
  {'alpha': 100.0, 'hidden_layer_sizes': (6,)} | val RMSE=0.5694 | val MAE=0.3794

All 9 models fitted on the primary specification.


In [28]:
'''
# 4.6 - Figure 4.1 training fit, four representative models
# Tree ensembles and the network can fit 117 training quarters closely while generalising
# poorly. Comparing training RMSE against the validation MAE from 4.5 shows which models
# are overfitting, which the evaluation-window figures in Section 5 cannot reveal.
# Validation error stays on MAE because hyperparameters were tuned on it; RMSE is the
# reporting metric.
SHOW = ['OLS', 'Random Forest', 'XGBoost', 'MLP']

fig = make_subplots(rows=2, cols=2, subplot_titles=SHOW,
                    vertical_spacing=0.13, horizontal_spacing=0.08)

for i, name in enumerate(SHOW):
    r, c = i // 2 + 1, i % 2 + 1
    res  = results[name]
    ix   = res['idx_train']
    fig.add_trace(go.Scatter(x=ix, y=res['y_train'], mode='lines', name='Actual',
                             line=dict(color=NAVY, width=1.7),
                             showlegend=(i == 0)), row=r, col=c)
    fig.add_trace(go.Scatter(x=ix, y=res['train_pred'], mode='lines', name='Fitted',
                             line=dict(color=TEAL, width=1.4, dash='dot'),
                             showlegend=(i == 0)), row=r, col=c)
    val = res['val_mae']
    fig.add_annotation(
        x=0.03, y=0.94, xref='x domain', yref='y domain', row=r, col=c,
        text=(f'train RMSE={rmse(res["y_train"], res["train_pred"]):.3f}'
              + ('' if np.isnan(val) else f' | val MAE={val:.3f}')),
        showarrow=False, font=dict(size=9, color=NAVY),
        bgcolor='rgba(255,255,255,0.8)', bordercolor=NAVY, borderwidth=0.7)

fig.update_layout(
    title=dict(text=('<b>Figure 4.1 - Training Fit by Model (1991 Q1 to 2020 Q4)</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     'A close training fit with a high validation error indicates '
                     'overfitting rather than skill. Validation MAE is shown because '
                     'tuning used MAE; RMSE is the reporting metric</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.97, yanchor='top'),
    template=TEMPLATE, height=620,
    legend=dict(orientation='h', yanchor='top', y=1.07, xanchor='center', x=0.5),
    margin=dict(t=120, b=45))
fig.update_yaxes(title_text='Delinquency Rate (%)', col=1)
fig.show()
'''

'\n# 4.6 - Figure 4.1 training fit, four representative models\n# Tree ensembles and the network can fit 117 training quarters closely while generalising\n# poorly. Comparing training RMSE against the validation MAE from 4.5 shows which models\n# are overfitting, which the evaluation-window figures in Section 5 cannot reveal.\n# Validation error stays on MAE because hyperparameters were tuned on it; RMSE is the\n# reporting metric.\nSHOW = [\'OLS\', \'Random Forest\', \'XGBoost\', \'MLP\']\n\nfig = make_subplots(rows=2, cols=2, subplot_titles=SHOW,\n                    vertical_spacing=0.13, horizontal_spacing=0.08)\n\nfor i, name in enumerate(SHOW):\n    r, c = i // 2 + 1, i % 2 + 1\n    res  = results[name]\n    ix   = res[\'idx_train\']\n    fig.add_trace(go.Scatter(x=ix, y=res[\'y_train\'], mode=\'lines\', name=\'Actual\',\n                             line=dict(color=NAVY, width=1.7),\n                             showlegend=(i == 0)), row=r, col=c)\n    fig.add_trace(go.Scatter(x

In [29]:
# 4.6 - Figure 4.1 training fit, all nine models
# Tree ensembles and the network can fit 117 training quarters closely while generalising
# poorly. Comparing training RMSE against validation RMSE identifies which models are
# memorising rather than learning, which the evaluation-window figures in Section 5 cannot
# reveal. Both figures are RMSE, the metric hyperparameters were also selected on. OLS has
# no hyperparameters and so no validation score.
ncols = 3
nrows = int(np.ceil(len(MODEL_NAMES) / ncols))

fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=MODEL_NAMES,
                    vertical_spacing=0.10, horizontal_spacing=0.06,
                    shared_yaxes=True)

for i, name in enumerate(MODEL_NAMES):
    r, c   = i // ncols + 1, i % ncols + 1
    res    = results[name]
    ix     = res['idx_train']
    is_ols = (name == 'OLS')

    fig.add_trace(go.Scatter(x=ix, y=res['y_train'], mode='lines', name='Actual',
                             line=dict(color=NAVY, width=1.6),
                             showlegend=(i == 0)), row=r, col=c)
    fig.add_trace(go.Scatter(x=ix, y=res['train_pred'], mode='lines', name='Fitted',
                             line=dict(color=TEAL if is_ols else BLUE,
                                       width=1.4, dash='dot'),
                             showlegend=(i == 0)), row=r, col=c)

    tr_rmse = rmse(res['y_train'], res['train_pred'])
    vr      = res['val_rmse']
    label   = (f'train {tr_rmse:.3f}' if np.isnan(vr)
               else f'train {tr_rmse:.3f} | val {vr:.3f} | gap {vr - tr_rmse:+.2f}')

    fig.add_annotation(
        x=0.03, y=0.95, xref='x domain', yref='y domain', row=r, col=c,
        text=label, showarrow=False, font=dict(size=8.5, color=NAVY),
        xanchor='left', bgcolor='rgba(255,255,255,0.85)',
        bordercolor=TEAL if is_ols else NAVY, borderwidth=0.7)

fig.update_layout(
    title=dict(text=('<b>Figure 4.1 - Training Fit by Model (1991 Q1 to 2020 Q4)</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     'Annotations give training and validation RMSE and the gap between '
                     'them. A small training error with a large gap indicates overfitting '
                     'rather than skill. OLS has no hyperparameters and so no validation '
                     'score</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.975, yanchor='top'),
    template=TEMPLATE, height=250 * nrows,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(t=135, b=45))
fig.update_yaxes(title_text='Delinquency Rate (%)', col=1)
fig.show()

gap_df = pd.DataFrame([
    {'Model': n,
     'Train RMSE': round(rmse(results[n]['y_train'], results[n]['train_pred']), 4),
     'Val RMSE': results[n]['val_rmse'],
     'Gap': (np.nan if np.isnan(results[n]['val_rmse']) else
             round(results[n]['val_rmse']
                   - rmse(results[n]['y_train'], results[n]['train_pred']), 4))}
    for n in MODEL_NAMES]).sort_values('Gap', ascending=False).set_index('Model')

print('Training and validation RMSE, sorted by gap. A large gap indicates overfitting;')
print('a negative gap indicates underfitting on the unusually flat 2017-2020 window.')
display(gap_df)

Training and validation RMSE, sorted by gap. A large gap indicates overfitting;
a negative gap indicates underfitting on the unusually flat 2017-2020 window.


,Train RMSE,Val RMSE,Gap
Model,,,
XGBoost,0.0095,1.0072,0.9977
Random Forest,0.5706,1.2340,0.6634
SVR,0.6846,1.2218,0.5372
Ridge,0.6089,1.0292,0.4203
Lasso,0.6105,1.0233,0.4128
Elastic Net,0.6105,1.0233,0.4128
KRR,0.5442,0.7227,0.1785
MLP,0.9426,0.5694,-0.3732
OLS,0.6087,NaN,NaN


In [30]:
# 4.7 - Figure 4.2 OLS residual diagnostics
ols_m = results['OLS']['model']
resid = ols_m.resid
ix_tr = results['OLS']['idx_train']
dw    = durbin_watson(resid)
lb    = acorr_ljungbox(resid, lags=[4, 8], return_df=True)
jb_p  = stats.jarque_bera(resid)[1]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
                    subplot_titles=['Residuals over time',
                                    'Residual distribution vs normal'])

fig.add_trace(go.Scatter(x=ix_tr, y=resid, mode='lines+markers', showlegend=False,
                         line=dict(color=BLUE, width=1.1), marker=dict(size=3)),
              row=1, col=1)
fig.add_hline(y=0, line_dash='dash', line_color=GREY, line_width=1, row=1, col=1)

fig.add_trace(go.Histogram(x=resid, nbinsx=25, histnorm='probability density',
                           marker_color=BLUE, opacity=0.6, showlegend=False),
              row=1, col=2)
grid = np.linspace(resid.min(), resid.max(), 200)
fig.add_trace(go.Scatter(x=grid, y=stats.norm.pdf(grid, resid.mean(), resid.std()),
                         mode='lines', line=dict(color=RED, width=1.5),
                         showlegend=False), row=1, col=2)

fig.update_layout(
    title=dict(text=('<b>Figure 4.2 - OLS Residual Diagnostics</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     f'DW={dw:.3f} | Ljung-Box p(4)={lb["lb_pvalue"].iloc[0]:.4f} | '
                     f'Jarque-Bera p={jb_p:.4f} | residuals run negative after 2011, '
                     'a level shift the specification cannot absorb</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.96, yanchor='top'),
    template=TEMPLATE, height=400, margin=dict(t=115, b=50))
fig.update_yaxes(title_text='Residual (pp)', row=1, col=1)
fig.update_yaxes(title_text='Density', row=1, col=2)
fig.show()

print(f'DW={dw:.4f} | LB p(4)={lb["lb_pvalue"].iloc[0]:.4f}, '
      f'p(8)={lb["lb_pvalue"].iloc[1]:.4f} | Jarque-Bera p={jb_p:.4f}')

DW=0.9358 | LB p(4)=0.0000, p(8)=0.0000 | Jarque-Bera p=0.1878


DW = 0.936 - Strong positive serial autocorrelation in residuals. Confirmed by the Ljung-Box tests below. This is expected for a delinquency rate model as defaults are persistent, and OLS residuals will inherit that persistence. It does not bias coefficients, but it invalidates plain standard errors, which is precisely why Newey-West HAC is used throughout.

Ljung-Box p(4) = p(8) = 0.000 - The null of no autocorrelation is decisively rejected at both 4 and 8 lags. This corroborates the DW reading and justifies the HAC correction. Also confirms that standard errors without HAC would be substantially underestimated, making t-statistics unreliable.

Jarque-Bera p = 0.188 - Fail to reject normality. The residual distribution is consistent with a Gaussian, which we can see visually in the histogram - reasonable bell shape, no heavy tails or extreme skew.

## Section 5 - Evaluation

### Two evaluation windows

Models are scored over two overlapping windows.

The full window covers 20 quarters, 2021 Q1 to 2025 Q4, the complete post-training period.
However the first six quarters overlap the COVID spline adjustment, so the target values there
were reconstructed rather than observed. Scoring a model against reconstructed values rewards
reproducing the spline's assumptions rather than predicting actual outcomes. Full-window
figures are reported for completeness but are not the primary performance claim.

The clean sub-window covers 14 quarters, 2022 Q3 to 2025 Q4, where the target is genuinely
observed. All primary performance claims, the DM tests and the confirmatory OLS test rest on
this window. It also aligns with the time-series track, which excludes the COVID period, so
the two Stage B notebooks report on comparable ground.

### Metrics reported

RMSE is primary, in percentage points of the delinquency rate. MAE is reported alongside.
Comparing them shows whether error is uniform or concentrated in a few quarters.

sMAPE is scale-free and bounded.

Skill against persistence expresses RMSE relative to a no-change forecast. Skill against the
mean does the same relative to the prevailing training mean. Both references are built from
training data only.

R2 out-of-sample uses squared error against persistence, following the Stage A convention, so
that the same column name means the same quantity in both stages. Three related columns on
three bases is deliberate rather than an inconsistency - two skill columns on RMSE for
comparability with the time-series track, and one on squared error for comparability with
Stage A.

Training RMSE is included so that a low training error paired with a high evaluation error
identifies overfitting directly, rather than leaving it to be inferred from two figures.

Ljung-Box p tests whether the forecast errors are serially correlated. A low value means the
errors follow a pattern the model failed to capture.

### The three benchmarks

Naive persistence holds the last observed value constant across the whole horizon. This is the
random-walk forecast and the hardest benchmark to beat for a persistent series. One caveat
worth stating: the anchor is 2020 Q4, which falls inside the spline window, so the benchmark
is itself a reconstructed value rather than an observation.

The long-run mean predicts the prevailing training average for every quarter. This is the
weakest benchmark and mainly serves as a floor.

The unemployment-only regression is the simplest defensible macroeconomic model and the usual
single-variable satellite specification in practice. If the seven-variable model cannot beat a
one-variable model, that is worth knowing.

### The Diebold-Mariano test

The DM test asks whether the difference in forecast accuracy between two models is larger
than would be expected from sampling variation. The null hypothesis is equal expected loss. A
negative statistic means the first model is more accurate.

The test works on the loss differential, the per-quarter difference in squared error between
the two forecasts. Because forecast errors from overlapping horizons are serially correlated,
the variance of that differential has to be estimated with a heteroskedasticity and
autocorrelation consistent estimator. This notebook uses Bartlett weights, which taper the
contribution of longer lags. An unweighted window is not guaranteed to give a positive
variance estimate and can fail silently.

The truncation lag follows the automatic bandwidth rule, four times the sample size over one
hundred raised to the power two-ninths, which depends on sample size rather than forecast
horizon. A horizon-based lag is not well defined in this design - with a single origin rather
than rolling origins the forecasts are one error path across horizons one to twenty rather
than a sample of h-step-ahead forecasts, and setting the lag to the window length collapses
the small-sample correction to exactly zero, forcing the statistic to zero regardless of the
data.

Power is low regardless. One error path across 14 correlated quarters carries far less
information than 14 independent comparisons. The practical consequence is that no comparison
reaches significance in either direction, including models that lose by wide margins. That is
a statement about the design, not about equivalence between the models. Read the skill columns
for magnitude and the p-values for confidence.

The sign test is reported alongside as a finite-sample companion. It counts the quarters in
which one forecast has the lower absolute error and compares that count against a coin flip.
It makes no distributional assumption, which makes it more reliable at this sample size,
though it discards information about the size of each error.

### One confirmatory test

Only one comparison is treated as confirmatory: OLS against naive persistence on the clean
sub-window, at an uncorrected 5 per cent level. Every other comparison in this notebook is
exploratory.

The reason is multiplicity. With nine models and one short window, the probability that at
least one beats the benchmark by chance is high. Varma and Simon (2006) formalise this - with
K candidates the probability that the minimum observed error understates the true error grows
as one minus one-half to the power K. Treating all nine comparisons as confirmatory would
overstate the evidence considerably. Designating one in advance, and reporting the rest as
exploratory, keeps the claim honest.

In [31]:
# 5.1 - Diebold-Mariano and sign test
def dm_test(y, f1, f2, lag=None, alpha=0.05):
    """Diebold-Mariano test under squared-error loss, Bartlett-weighted HAC variance.
    H0: equal expected loss. A negative statistic means f1 is more accurate than f2.

    The truncation lag follows the automatic bandwidth 4*(T/100)^(2/9), which depends on
    sample size rather than forecast horizon. A horizon-based lag is undefined here: with
    one origin rather than rolling origins, setting it to the window length collapses the
    small-sample correction to zero. Power is low either way, so the sign test is the more
    reliable companion.
    """
    y, f1, f2 = np.asarray(y), np.asarray(f1), np.asarray(f2)
    # Squared-error loss, matching RMSE as the primary reporting metric.
    d = (y - f1) ** 2 - (y - f2) ** 2
    T = len(d)

    if np.allclose(d, 0):
        return dict(stat=np.nan, p=np.nan, sig=False, lag=0, note='self-comparison')

    if lag is None:
        lag = int(np.floor(4 * (T / 100) ** (2 / 9)))
    lag = max(0, min(lag, T - 2))

    e   = d - d.mean()
    var = np.dot(e, e) / T
    for k in range(1, lag + 1):
        var += 2 * (1 - k / (lag + 1)) * np.dot(e[k:], e[:-k]) / T
    var /= T

    if var <= 0:
        return dict(stat=np.nan, p=np.nan, sig=False, lag=lag,
                    note='non-positive HAC variance')

    stat = d.mean() / np.sqrt(var)
    p    = 2 * (1 - stats.t.cdf(abs(stat), df=T - 1))
    return dict(stat=round(stat, 4), p=round(p, 4), sig=p < alpha, lag=lag, note='')

def sign_test(y, f1, f2):
    """Counts quarters where f1 has lower absolute error than f2; under H0 the count is
    Binomial(T, 0.5). Returns NaN for a self-comparison, where all differentials are zero
    and the count would be spuriously significant."""
    d = np.abs(y - np.asarray(f1)) - np.abs(y - np.asarray(f2))
    T = len(d)
    if np.allclose(d, 0):
        return 0, T, np.nan
    wins = int((d < 0).sum())
    p = min(1.0, 2 * min(stats.binom.cdf(wins, T, 0.5),
                         1 - stats.binom.cdf(wins - 1, T, 0.5)))
    return wins, T, round(p, 4)

print(f'DM automatic lag: {int(np.floor(4*(14/100)**(2/9)))} at T=14, '
      f'{int(np.floor(4*(20/100)**(2/9)))} at T=20')

DM automatic lag: 2 at T=14, 2 at T=20


In [32]:
# 5.2 - benchmarks
X_ev, y_ev, idx_ev = mats_primary['eval']
y_tr               = mats_primary['train'][1]
idx_ev             = pd.DatetimeIndex(idx_ev)
clean_mask         = (idx_ev >= CLEAN_START) & (idx_ev <= CLEAN_END)

# All three are constructible at the 2020 Q4 origin.
bm_persist = np.full(len(y_ev), y_tr[-1])      # random walk: last observed value held
bm_mean    = np.full(len(y_ev), y_tr.mean())   # prevailing training mean

# Unemployment-only regression: the simplest defensible macro model, and the usual
# single-variable satellite specification in practice.
u_tr = df_train[['us_unemployment_L0', TARGET, 'covid_dummy']].dropna()
ols_u = sm.OLS(u_tr[TARGET].values,
               sm.add_constant(u_tr[['us_unemployment_L0', 'covid_dummy']].values,
                               has_constant='add')
               ).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
bm_unemp = ols_u.predict(sm.add_constant(
    df_eval.loc[idx_ev, ['us_unemployment_L0', 'covid_dummy']].values, has_constant='add'))

BENCHMARKS = {'Naive persistence': bm_persist,
              'Long-run mean': bm_mean,
              'Unemployment only': bm_unemp}

# The persistence anchor falls inside the spline window, so the benchmark the confirmatory
# test runs against is itself a reconstructed value rather than an observation.
anchor_date = pd.Timestamp(mats_primary['train'][2][-1])
print(f'Persistence anchor: {y_tr[-1]:.3f}% at {q(anchor_date)} '
      f'(inside spline window: '
      f'{pd.Timestamp(COVID_START) <= anchor_date <= pd.Timestamp(COVID_END)})')
print(f'Long-run mean: {y_tr.mean():.3f}%')
print(f'Unemployment-only R2: {ols_u.rsquared:.3f}')
print(f'Clean sub-window: {clean_mask.sum()} of {len(idx_ev)} evaluation quarters')

Persistence anchor: 2.261% at 2020 Q4 (inside spline window: True)
Long-run mean: 3.860%
Unemployment-only R2: 0.150
Clean sub-window: 14 of 20 evaluation quarters


In [33]:
# 5.3 - Table 5.1 metrics on both windows
def metric_row(label, window, y, f, ref_persist, ref_mean):
    wins, T, sign_p = sign_test(y, f, ref_persist)
    dm  = dm_test(y, f, ref_persist)
    err = np.asarray(y) - np.asarray(f)
    lb  = acorr_ljungbox(err, lags=[min(4, len(err) // 2)], return_df=True)
    tr  = (round(rmse(results[label]['y_train'], results[label]['train_pred']), 4)
           if label in results else np.nan)
    return {'Model': label, 'Window': window,
            'Train RMSE': tr,
            'RMSE': round(rmse(y, f), 4), 'MAE': round(mae(y, f), 4),
            'sMAPE': round(smape(y, f), 2),
            'Skill vs persist': round(skill(y, f, ref_persist, 'rmse'), 4),
            'Skill vs mean': round(skill(y, f, ref_mean, 'rmse'), 4),
            'LB p': round(lb['lb_pvalue'].iloc[0], 4),
            'DM stat': dm['stat'], 'DM p': dm['p'],
            'DM sig': 'Yes' if dm['sig'] else 'No',
            'Sign': f'{wins}/{T}', 'Sign p': sign_p,
            'R2_oos (vs persist)': round(skill(y, f, ref_persist, 'mse'), 4),}

WINDOWS = [('Full (20Q)', slice(None)), ('Clean (14Q)', clean_mask)]

rows = []
for name in MODEL_NAMES:
    r = results[name]
    for wlabel, m in WINDOWS:
        rows.append(metric_row(name, wlabel, r['y_eval'][m], r['eval_pred'][m],
                               bm_persist[m], bm_mean[m]))
for bname, bpred in BENCHMARKS.items():
    for wlabel, m in WINDOWS:
        rows.append(metric_row(bname, wlabel, y_ev[m], bpred[m],
                               bm_persist[m], bm_mean[m]))

metrics_df = pd.DataFrame(rows)
clean_tbl  = (metrics_df[metrics_df['Window'] == 'Clean (14Q)']
              .sort_values('RMSE').set_index('Model'))

print('Table 5.1 - Clean test-set (2022 Q3 to 2025 Q4), primary reporting window')
print('Sorted by RMSE, the primary metric. Skill columns use RMSE; R2_oos uses squared '
      'error, matching the Stage A convention.')
display(clean_tbl[['Train RMSE', 'RMSE', 'MAE', 'sMAPE', 'Skill vs persist',
                   'Skill vs mean', 'R2_oos (vs persist)', 'DM stat', 'DM p', 'DM sig',
                   'Sign', 'Sign p']])

n_clean = int(clean_mask.sum())
print(f'\nDM tests compare each row against naive persistence over {n_clean} quarters '
      f'from a single origin.')
print('No comparison reaches significance in either direction, including models that '
      'lose by wide margins.')
print('This reflects the power of the design rather than equivalence between the '
      'models - read the skill columns for magnitude, the p-values for confidence.')
print('NaN rows are self-comparisons (persistence against itself) where every loss '
      'differential is zero.')

Table 5.1 - Clean test-set (2022 Q3 to 2025 Q4), primary reporting window
Sorted by RMSE, the primary metric. Skill columns use RMSE; R2_oos uses squared error, matching the Stage A convention.


,Train RMSE,RMSE,MAE,sMAPE,Skill vs persist,Skill vs mean,R2_oos (vs persist),DM stat,DM p,DM sig,Sign,Sign p
Model,,,,,,,,,,,,
MLP,0.9426,0.6464,0.5079,16.92,0.0870,0.3816,0.1663,-0.2681,0.7928,No,9/14,0.4240
Naive persistence,NaN,0.7080,0.6451,24.44,0.0000,0.3227,0.0000,NaN,NaN,No,0/14,NaN
Unemployment only,NaN,0.7417,0.6746,21.71,-0.0477,0.2904,-0.0976,0.1513,0.8821,No,9/14,0.4240
KRR,0.5442,0.7647,0.6155,19.45,-0.0802,0.2684,-0.1668,0.2602,0.7988,No,8/14,0.7905
XGBoost,0.0095,0.9120,0.7005,21.42,-0.2883,0.1275,-0.6597,0.8381,0.4171,No,7/14,1.0000
SVR,0.6846,1.0167,0.7124,21.91,-0.4361,0.0273,-1.0625,0.7246,0.4815,No,9/14,0.4240
Long-run mean,NaN,1.0453,0.9846,29.90,-0.4765,0.0000,-1.1799,1.3124,0.2121,No,5/14,0.4240
Elastic Net,0.6105,1.0896,0.8347,24.80,-0.5391,-0.0424,-1.3689,1.1387,0.2754,No,7/14,1.0000
Lasso,0.6105,1.0896,0.8347,24.80,-0.5391,-0.0424,-1.3689,1.1387,0.2754,No,7/14,1.0000



DM tests compare each row against naive persistence over 14 quarters from a single origin.
No comparison reaches significance in either direction, including models that lose by wide margins.
This reflects the power of the design rather than equivalence between the models - read the skill columns for magnitude, the p-values for confidence.
NaN rows are self-comparisons (persistence against itself) where every loss differential is zero.


In [34]:
# 5.3b - sensitivity of a challenger to the hyperparameter selection criterion
# The tuning criterion should not materially change out-of-sample accuracy for a fixed
# model class and grid. This refits one challenger under both criteria to test that.
def tune_and_score(estimator, grid, mats, criterion):
    X_tr, y_tr, _  = mats['train']
    X_tu, y_tu, _  = mats['tune']
    X_va, y_va, _  = mats['val']
    X_ev, y_ev, _  = mats['eval']
    sc = StandardScaler().fit(X_tr)
    t  = lambda X: sc.transform(X)

    best, bp = np.inf, {}
    for params in ParameterGrid(grid):
        est = clone(estimator).set_params(**params).fit(t(X_tu), y_tu)
        pred = est.predict(t(X_va))
        s = (mean_absolute_error(y_va, pred) if criterion == 'mae'
             else float(np.sqrt(mean_squared_error(y_va, pred))))
        if s < best:
            best, bp = s, params

    final = clone(estimator).set_params(**bp).fit(t(X_tr), y_tr)
    ev    = final.predict(t(X_ev))
    return bp, round(rmse(y_ev[clean_mask], ev[clean_mask]), 4)

TEST_MODEL = 'KRR'
est, grid = GRIDS[TEST_MODEL]
rows = []
for crit in ['mae', 'rmse']:
    bp, r = tune_and_score(est, grid, mats_primary, crit)
    rows.append({'Tuned on': crit.upper(), 'Selected params': str(bp),
                 'Clean RMSE': r,
                 'Skill vs persist': round(1 - r / rmse(y_ev[clean_mask],
                                                        bm_persist[clean_mask]), 4)})

sens = pd.DataFrame(rows).set_index('Tuned on')
print(f'{TEST_MODEL} under two hyperparameter selection criteria, clean sub-window')
display(sens)
delta = sens['Clean RMSE'].iloc[1] / sens['Clean RMSE'].iloc[0] - 1
print(f'Change in test RMSE from the selection criterion alone: {delta:+.1%}')
print('Model class, candidate grid, training data and evaluation window are identical.')

KRR under two hyperparameter selection criteria, clean sub-window


,Selected params,Clean RMSE,Skill vs persist
Tuned on,,,
MAE,"{'alpha': 0.01, 'gamma': 0.01}",0.6352,0.1028
RMSE,"{'alpha': 0.1, 'gamma': 0.01}",0.7647,-0.0801


Change in test RMSE from the selection criterion alone: +20.4%
Model class, candidate grid, training data and evaluation window are identical.


In [35]:
# 5.4 - Table 5.2 per-horizon accuracy
# Cumulative error over the first h quarters from the 2020 Q4 origin. Shows whether
# accuracy degrades with horizon, which a single window-level figure conceals.
h_rows = []
for name in MODEL_NAMES + list(BENCHMARKS):
    pred = results[name]['eval_pred'] if name in results else BENCHMARKS[name]
    row = {'Model': name}
    for h in HORIZONS:
        row[f'h={h}'] = round(rmse(y_ev[:h], pred[:h]), 4)
    h_rows.append(row)

horizon_df = pd.DataFrame(h_rows).set_index('Model')
print('Table 5.2 - RMSE by forecast horizon (cumulative from 2020 Q4 origin, full window)')
display(horizon_df)

Table 5.2 - RMSE by forecast horizon (cumulative from 2020 Q4 origin, full window)


,h=1,h=4,h=8,h=12,h=20
Model,,,,,
OLS,0.0364,1.2516,1.3225,1.3601,1.0999
Ridge,0.0102,1.2149,1.3026,1.3465,1.0894
Lasso,0.0032,1.0894,1.2299,1.2883,1.0408
Elastic Net,0.0032,1.0894,1.2299,1.2883,1.0408
KRR,0.2119,0.6113,0.7162,0.8454,0.6960
SVR,1.8018,2.1707,2.2077,1.9164,1.4916
Random Forest,1.5650,2.0312,1.7817,1.6954,1.3416
XGBoost,1.1873,1.7596,1.4700,1.3838,1.1596
MLP,0.1220,0.7689,0.8951,0.8434,0.6730


In [36]:
# 5.5 - Figure 5.1 forecasts against actual
ncols = 3
nrows = int(np.ceil(len(MODEL_NAMES) / ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=MODEL_NAMES,
                    vertical_spacing=0.11, horizontal_spacing=0.07)

for i, name in enumerate(MODEL_NAMES):
    r, c = i // ncols + 1, i % ncols + 1
    res = results[name]
    ix  = pd.DatetimeIndex(res['idx_eval'])
    fig.add_vrect(x0=str(ix[0].date()), x1=CLEAN_START, row=r, col=c,
                  fillcolor='rgba(230,126,34,0.30)', line_width=0)
    fig.add_trace(go.Scatter(x=ix, y=res['y_eval'], mode='lines', name='Actual',
                             line=dict(color=NAVY, width=2),
                             showlegend=(i == 0)), row=r, col=c)
    fig.add_trace(go.Scatter(x=ix, y=res['eval_pred'], mode='lines+markers',
                             name='Forecast', line=dict(color=TEAL, width=1.6, dash='dash'),
                             marker=dict(size=3), showlegend=(i == 0)), row=r, col=c)
    fig.add_trace(go.Scatter(x=ix, y=bm_persist, mode='lines', name='Persistence',
                             line=dict(color=GREY, width=1, dash='dot'),
                             showlegend=(i == 0)), row=r, col=c)

'''
fig.update_layout(
    title=dict(text=('<b>Figure 5.1 - Evaluation Window Forecasts (2021 Q1 to 2025 Q4)</b>',
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.975, yanchor='top'),
    template=TEMPLATE, height=260 * nrows,
    legend=dict(orientation='h', yanchor='top', y=1.06, xanchor='center', x=0.5),
    margin=dict(t=125, b=40))
'''
fig.update_layout(
    title=dict(text='<b>Figure 5.1 - Evaluation Window Forecasts (2021 Q1 to 2025 Q4)</b>',
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.975, yanchor='top'),
    template=TEMPLATE, height=260 * nrows,
    legend=dict(orientation='h', yanchor='top', y=1.1, xanchor='center', x=0.5),
    margin=dict(t=125, b=40))
fig.show()

In [37]:
# 5.6 - pre-registered confirmatory test
# One confirmatory comparison at uncorrected 5%: OLS against naive persistence on the
# clean sub-window. Every other comparison in this notebook is exploratory. With nine
# models the chance that at least one beats the benchmark by luck is high, so treating all
# of them as confirmatory would overstate the evidence (Varma and Simon 2006).
y_c   = y_ev[clean_mask]
ols_c = results['OLS']['eval_pred'][clean_mask]
p_c   = bm_persist[clean_mask]

dm_c = dm_test(y_c, ols_c, p_c)
w, T, sp = sign_test(y_c, ols_c, p_c)

print('=' * 62)
print('PRE-REGISTERED CONFIRMATORY TEST')
print('OLS vs naive persistence | clean 14Q sub-window | alpha=0.05 | squared-error loss')
print('=' * 62)
print(f'OLS RMSE         : {rmse(y_c, ols_c):.4f}pp')
print(f'Persistence RMSE : {rmse(y_c, p_c):.4f}pp')
print(f'OLS MAE          : {mae(y_c, ols_c):.4f}pp  (secondary)')
print(f'Skill vs persist : {skill(y_c, ols_c, p_c, "rmse"):+.4f}  (RMSE basis)')
print(f'DM statistic    : {dm_c["stat"]} (Bartlett HAC, lag={dm_c["lag"]})')
print(f'DM p-value      : {dm_c["p"]}  ->  '
      f'{"SIGNIFICANT" if dm_c["sig"] else "not significant"}')
print(f'Sign test       : OLS wins {w}/{T}, p={sp}')
print('-' * 62)
if dm_c['sig'] and rmse(y_c, ols_c) < rmse(y_c, p_c):
    print('OLS significantly outperforms naive persistence.')
elif dm_c['sig']:
    print('Difference is significant, but persistence is the more accurate forecast.')
else:
    print('No significant difference from naive persistence. OLS remains the')
    print('deliverable on governance grounds; the null result is reported as such.')
print('=' * 62)

PRE-REGISTERED CONFIRMATORY TEST
OLS vs naive persistence | clean 14Q sub-window | alpha=0.05 | squared-error loss
OLS RMSE         : 1.1213pp
Persistence RMSE : 0.7080pp
OLS MAE          : 0.8566pp  (secondary)
Skill vs persist : -0.5839  (RMSE basis)
DM statistic    : 1.2135 (Bartlett HAC, lag=2)
DM p-value      : 0.2465  ->  not significant
Sign test       : OLS wins 7/14, p=1.0
--------------------------------------------------------------
No significant difference from naive persistence. OLS remains the
deliverable on governance grounds; the null result is reported as such.


Two separate points:

"No significant difference from naive persistence" - The DM p-value is 0.25, above the pre-registered 5% threshold, so the null of equal predictive accuracy is not rejected. OLS is actually worse than persistence on RMSE (1.12 vs 0.71), but the test cannot confirm that difference is real rather than noise at T=14.

"OLS remains the deliverable on governance grounds" - Despite losing to persistence in this window, OLS is still the model we go with. Naive persistence - just carrying forward last quarter's delinquency rate - cannot satisfy regulatory requirements as it has no economic mechanism, cannot be stress-tested under adverse scenarios, and cannot be explained to auditors or regulators. OLS with signed, interpretable coefficients can do all of those things. This null result is reported transparently rather than being treated as a failure that needs to be explained away.

### SHAP attribution, and how to read it

This is the first use of SHAP in the project, so this subsection explains what it is, why it
is here, and how to interpret the output.

The evaluation above tells us which models are most accurate. It does not tell us whether an
accurate challenger is using the same economic information as OLS in a more flexible way, or
whether it is relying on something quite different.

That distinction matters for the deliverable. If a challenger's variable importance structure
broadly agrees with OLS, then the OLS coefficients remain a fair summary of what drives the
forecast, and the transparency argument for OLS holds without much cost. If the challenger
relies on a substantially different structure, then OLS is not merely less accurate but is
describing a different relationship, and that is a more serious limitation.

Under IFRS 9 institutions must be able to explain and challenge their model outputs, so the
question of whether accuracy and interpretability can be reconciled is an important one.

#### What is SHAP?

SHAP stands for SHapley Additive exPlanations (Lundberg and Lee 2017). It decomposes any
model's individual prediction into contributions from each input variable.

The idea comes from cooperative game theory. Treat each feature as a player in a game and the
prediction as the prize to be shared. The Shapley value of a player is its average marginal
contribution across every possible subset of the other players. Applied to a model, the SHAP
value of a feature for one observation is how much that feature moved the prediction away from
the model's baseline, averaged over all the ways the other features could have been present or
absent.

The decomposition is additive:

    prediction for quarter i = baseline + shap_1(i) + shap_2(i) + ... + shap_k(i)

where the baseline is the model's average prediction across the background data. Because the
contributions sum exactly to the prediction, SHAP values are directly comparable to OLS
coefficient contributions in both direction and relative size, even when the underlying model
is nonlinear.

This notebook checks that additivity property explicitly. The reported additivity error is the
largest discrepancy between the reconstructed sum and the model's actual prediction. It should
be effectively zero. A materially non-zero value means the explainer has not converged and the
attribution should not be trusted.

#### Two different computations for two different models

For OLS the SHAP value has a closed form and no approximation is needed. For a linear model
the contribution of feature j in quarter i is simply

    shap_j(i) = beta_j (x_j(i) - mean(x_j))

that is, the coefficient multiplied by how far the feature sits from its own average. Both
terms are on the original scale, so the result is in percentage points of delinquency and can
be read directly. Computing this analytically avoids an unnecessary approximation and is
exact.

For the challenger, which was fitted on standardised inputs and may be nonlinear, SHAP values
are approximated by KernelExplainer. This is model-agnostic - it treats the model as a black
box and estimates Shapley values by evaluating it on perturbed inputs. It is slower than the
exact methods available for trees or linear models, which is why the background reference set
is reduced by k-means clustering to keep runtime manageable.

#### How to read the beeswarm panel

Each dot is one training quarter. Its horizontal position is the SHAP value, how far that
feature pushed the prediction above or below the baseline in that quarter, measured in
percentage points. Its colour shows whether the feature value itself was high or low in that
quarter.

A clean pattern of high values on the right and low values on the left means high values of
the feature raise the prediction. This is equivalent to a positive coefficient and OLS could
capture it. The mirror pattern, high on the left, is equivalent to a negative coefficient.

Mixed colours on the same side, or a wide scattered cloud, indicates a nonlinear effect, an
interaction with another variable, or a relationship that differs across economic regimes.
This is the signature of structure that OLS cannot represent, and it is the mechanism through
which a challenger earns any accuracy advantage.

The horizontal width of a feature's cloud shows how much prediction variation that feature is
responsible for. A wide cloud means large and variable effects; a narrow cloud clustered near
zero means the model barely uses the variable.

#### How to read the ranking comparison panel

The bar chart shows mean absolute SHAP value per variable, normalised so each model's
contributions sum to one. This puts both models on the same footing: a share of total
attributed importance rather than an absolute magnitude.

Bars of similar length mean the two models assign the variable comparable importance. Where
they differ substantially, the two models are using that variable differently.

The accompanying table reports each model's rank for every variable and the gap between the
ranks, along with the Spearman rank correlation across all variables. A high rank correlation
means the challenger's importance structure is broadly consistent with OLS even if its
functional form is more complex, which supports treating the OLS coefficients as a reasonable
description of what drives the forecast. A low correlation would mean the two models disagree
about what matters.

#### Caveats

SHAP explains the model, not the world. A high SHAP value means the model relies on that
variable, not that the variable causes defaults.

SHAP values are computed on the training data here, so they describe how the model behaves
over the period it was fitted on.

Attribution among correlated variables is not uniquely determined. When two variables carry
overlapping information, how the credit is split between them depends on the background
distribution used. With three real-activity measures in this specification, the individual
attributions among them should be read with that in mind, while the total across the group is
more stable.

Finally, which challenger gets explained here depends on the accuracy ranking, which is itself
a post-hoc choice on the test window. This analysis is exploratory and does not affect the
deliverable.

In [38]:
# 5.7 - Table 5.3 SHAP decomposition, OLS against the two leading challengers
# Two challengers rather than one. A single comparison cannot distinguish a genuine
# difference in what the model relies on from an idiosyncrasy of one fitted model. If both
# challengers depart from OLS in the same direction, that is evidence about the
# specification; if they disagree with each other as much as with OLS, the attribution is
# unstable and no conclusion should be drawn.
import shap

ranked             = clean_tbl.drop(index=['OLS'] + list(BENCHMARKS),
                                    errors='ignore').index
best_ml, second_ml = ranked[0], ranked[1]
CHALLENGERS        = [best_ml, second_ml]
ALL_EXPLAINED      = ['OLS'] + CHALLENGERS

X_tr, y_tr = mats_primary['train'][0], mats_primary['train'][1]
feats      = FEATURES_EQ511

# OLS is linear, so its SHAP values are exact and analytic: beta_j (x_j - mean(x_j)).
# Both terms are on the original scale, giving contributions in percentage points.
beta      = np.asarray(results['OLS']['model'].params[1:])
shap_vals = {'OLS': (X_tr - X_tr.mean(axis=0)) * beta}

# Challengers were fitted on scaled inputs and may be nonlinear, so KernelExplainer is used.
# It is model-agnostic and treats each model as a black box. The background set is reduced
# by k-means to keep runtime manageable. Output remains in percentage points.
for name in CHALLENGERS:
    Xs   = results[name]['scaler'].transform(X_tr)
    expl = shap.KernelExplainer(results[name]['model'].predict, shap.kmeans(Xs, 25))
    sv   = np.asarray(expl.shap_values(Xs, silent=True))
    # Additivity: baseline plus contributions must reconstruct the prediction. A materially
    # non-zero error means the explainer has not converged and the attribution is unreliable.
    err  = np.abs(sv.sum(axis=1) + expl.expected_value
                  - results[name]['train_pred']).max()
    print(f'{name:16s} additivity error {err:.6f}pp')
    shap_vals[name] = sv

def norm(v):
    v = np.clip(v, 0, None)
    return v / v.sum()

imp = pd.DataFrame({'Variable': feats})
for m in ALL_EXPLAINED:
    imp[m] = norm(np.abs(shap_vals[m]).mean(axis=0)).round(4)
for m in ALL_EXPLAINED:
    imp[f'Rank {m}'] = imp[m].rank(ascending=False).astype(int)
for m in CHALLENGERS:
    imp[f'Gap {m}'] = (imp['Rank OLS'] - imp[f'Rank {m}']).abs()
imp = imp.sort_values('OLS', ascending=False)

print(f'\nTable 5.3 - Mean absolute SHAP by variable, normalised to sum to one')
print(f'Challengers ranked on RMSE: {best_ml} ({clean_tbl.loc[best_ml, "RMSE"]:.4f}) '
      f'then {second_ml} ({clean_tbl.loc[second_ml, "RMSE"]:.4f}), '
      f'against OLS ({clean_tbl.loc["OLS", "RMSE"]:.4f}).')
display(imp.set_index('Variable'))

# Pairwise rank agreement. The challenger-to-challenger figure is the control: if the two
# challengers agree with each other no more than either agrees with OLS, the differences
# are noise rather than a shared departure from the linear specification.
print('Spearman rank correlation between importance orderings:')
pairs = [('OLS', best_ml), ('OLS', second_ml), (best_ml, second_ml)]
rho_tbl = {}
for a, b in pairs:
    r = stats.spearmanr(imp[a], imp[b]).statistic
    rho_tbl[(a, b)] = r
    print(f'  {a:16s} vs {b:16s} {r:+.3f}')
rho = rho_tbl[('OLS', best_ml)]

MLP              additivity error 0.000000pp
KRR              additivity error 0.000000pp

Table 5.3 - Mean absolute SHAP by variable, normalised to sum to one
Challengers ranked on RMSE: MLP (0.6464) then KRR (0.7647), against OLS (1.1213).


,OLS,MLP,KRR,Rank OLS,Rank MLP,Rank KRR,Gap MLP,Gap KRR
Variable,,,,,,,,
us_credit_qoq_growth_L6,0.2777,0.3003,0.2881,1,1,1,0,0
us_unemployment_L0,0.1533,0.1357,0.1790,2,4,2,2,0
us_house_price_yoy_L3,0.1531,0.2282,0.1485,3,2,3,1,0
us_consumer_confidence_L2,0.1395,0.1423,0.1213,4,3,4,1,0
us_gdp_yoy_growth_L2,0.1115,0.0040,0.1002,5,8,5,3,0
us_cpi_L0,0.0776,0.1142,0.0764,6,5,6,1,0
us_indprod_yoy_L3,0.0444,0.0126,0.0384,7,7,8,0,1
covid_dummy,0.0429,0.0626,0.0480,8,6,7,2,1


Spearman rank correlation between importance orderings:
  OLS              vs MLP              +0.762
  OLS              vs KRR              +0.976
  MLP              vs KRR              +0.786


In [39]:
# 5.8 - Figure 5.2 importance shares
# How each model divides its total attributed importance across the regressors. Bars of
# similar length mean the models weight that variable comparably; a variable where one bar
# is much longer is one the models use differently. The rank agreement figures in the
# subtitle quantify the overall similarity.
order = list(imp['Variable'])[::-1]
short = [f.replace('us_', '').replace('_', ' ') for f in order]
COLS  = {'OLS': NAVY, best_ml: TEAL, second_ml: AMBER}

fig = go.Figure()
for m in ALL_EXPLAINED:
    vals = imp[m][::-1]
    fig.add_trace(go.Bar(
        x=vals, y=short, orientation='h', name=m,
        marker_color=COLS[m], opacity=0.9,
        text=[f'{v:.3f}' for v in vals], textposition='outside',
        textfont=dict(size=8.5, color=GREY), cliponaxis=False))

fig.update_layout(
    title=dict(text=('<b>Figure 5.2 - Share of Attributed Importance by Model</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     f'Rank agreement with OLS: {rho_tbl[("OLS", best_ml)]:+.2f} for '
                     f'{best_ml}, {rho_tbl[("OLS", second_ml)]:+.2f} for {second_ml}. '
                     f'The two challengers agree with each other at '
                     f'{rho_tbl[(best_ml, second_ml)]:+.2f}</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.96, yanchor='top'),
    xaxis=dict(title='Share of total attributed importance',
               range=[0, float(imp[ALL_EXPLAINED].max().max()) * 1.16],
               showgrid=True, gridcolor='rgba(0,0,0,0.07)'),
    yaxis=dict(showgrid=False, ticklabelposition='outside'),
    template=TEMPLATE, height=640, barmode='group',
    bargap=0.28, bargroupgap=0.06,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(t=125, b=60, l=185, r=40))
fig.show()

In [40]:
# 5.9 - Figure 5.3 per-quarter contributions for both challengers
# One dot per training quarter, coloured by the value of the variable in that quarter.
# A clean colour gradient across the horizontal axis means the effect is monotone and a
# linear model could capture it. Mixed colours on the same side indicate a nonlinear
# effect or an interaction, which is the mechanism by which a challenger can outperform.
pos = {f: i for i, f in enumerate(order)}
rng = np.random.RandomState(SEED)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.05, shared_yaxes=True,
                    subplot_titles=[f'{m}: per-quarter contributions'
                                    for m in CHALLENGERS])

for k, name in enumerate(CHALLENGERS):
    sv_all = shap_vals[name]
    for f in order:
        j  = feats.index(f)
        xv = X_tr[:, j]
        colour = (xv - xv.min()) / (np.ptp(xv) if np.ptp(xv) else 1)
        fig.add_trace(go.Scatter(
            x=sv_all[:, j], y=pos[f] + rng.uniform(-0.22, 0.22, len(sv_all)),
            mode='markers', showlegend=False,
            marker=dict(size=4.2, color=colour, colorscale='RdYlBu_r', opacity=0.75,
                        showscale=(k == 1 and f == order[-1]),
                        colorbar=dict(title=dict(text='feature value', font=dict(size=9),
                                                 side='top'),
                                      orientation='h', x=0.5, xanchor='center',
                                      y=-0.16, yanchor='top', len=0.3, thickness=9,
                                      tickvals=[0, 1], ticktext=['low', 'high'],
                                      tickfont=dict(size=8))),
            hovertemplate='%{x:.3f}pp<extra></extra>'), row=1, col=k + 1)
    fig.add_vline(x=0, line_dash='dash', line_color=GREY, line_width=1, row=1, col=k + 1)
    fig.update_xaxes(title_text='SHAP value (pp)', row=1, col=k + 1)

fig.update_yaxes(tickmode='array', tickvals=list(range(len(order))),
                 ticktext=short, row=1, col=1)
fig.update_layout(
    title=dict(text=('<b>Figure 5.3 - Per-Quarter SHAP Contributions</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     'Colour shows whether the variable was high or low in that quarter. '
                     'A clean gradient means a monotone effect; mixed colours on one side '
                     'indicate nonlinearity.</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.96, yanchor='top'),
    template=TEMPLATE, height=500, margin=dict(t=120, b=115, l=150))
fig.show()

## Section 6 - Winner Selection, Production Forecast and Export

### Why OLS is the deliverable regardless of the evaluation results

OLS is the pre-registered primary model. The decision that it would be the regulatory
deliverable was made before any evaluation results were seen, not after. Two reasons.

First, with nine models and one short evaluation window, the probability that at least one
outperforms OLS by chance is high. Selecting the best performer after seeing the results would
leave no reliable way to distinguish genuine skill from luck on this particular 14-quarter
window. Pre-registering OLS separates the confirmatory question, does OLS beat persistence,
from the exploratory one, which model happened to score best.

Second, IFRS 9 model governance requires that institutions explain and justify their model
outputs. An OLS coefficient states how much the delinquency rate changes per unit change in a
macroeconomic variable, holding the others constant, which can be communicated to an auditor
or regulator. A Random Forest or a neural network does not offer that. Any loss of predictive
accuracy from choosing OLS has to be weighed against this gain in transparency (Bank and Eder
2021, Section 9.1).

If the evaluation shows OLS does not significantly beat naive persistence, that result is
stated as a limitation rather than concealed by switching to whichever model happened to
perform better.

### What the production forecast represents

The production model refits OLS on the full history to 2025 Q4 rather than the training block
alone, so the coefficient estimates use the most recent five years including the post-COVID
recovery and the inflation shock. The inputs for the 20-quarter projection are the Stage A
macroeconomic forecasts, each of which is the winning forecast for that variable from the
Stage A competition.

The resulting delinquency path from 2026 Q1 to 2030 Q4 is the primary output of the project.
In a full IFRS 9 implementation this path would feed a scenario-weighting calculation
alongside upside and downside macroeconomic paths to produce a probability-weighted Expected
Credit Loss estimate. That final step is outside the scope of this project; the satellite
model built here is the input it would require.

### Reading the OLS coefficient table

The intercept has no standalone economic meaning. The regressors never take the value zero
simultaneously, and one of them, consumer confidence, is an index level around 100 rather than
a growth rate, so its coefficient contributes a large negative amount that the intercept
offsets. A large intercept is therefore an artefact of variable scaling rather than a
prediction.

Each remaining coefficient is the change in the delinquency rate in percentage points per
one-unit change in that variable at the stated lag, holding the others constant. Standard
errors are Newey-West with four lags, one year at quarterly frequency, which corrects
inference for the serial correlation visible in Figure 4.2.

The prior and match columns repeat the sign check from Section 2 on the production sample.
Reading them together with the nested trace in Section 2 matters - a coefficient that
contradicts its prior only in the full specification, and only becomes significant there, is
showing a collinearity artefact rather than an economic effect, and should not be interpreted
in isolation.

A general caution follows from that. Where several regressors share a transmission channel,
individual partial effects are not separately identified. The functional form remains
inspectable and the forecast remains decomposable, which is what the transparency argument for
OLS actually requires, but attributing a marginal effect to any single collinear regressor is
not supportable.

### When the forecast stops using real data

Table 6.2 records, for each regressor, the quarter at which its input switches from observed
data to a Stage A projection.

A lagged variable draws on real data for its first few forecast quarters. A variable at lag 6
supplies observed values for the first six quarters of the forecast, because forecasting
2026 Q1 with a lag-6 column reaches back to 2024 Q3, which is published data. A variable at
lag 0 depends on Stage A projections from the first forecast quarter onward.

This turns a general statement about uncertainty into a specific one. The earliest forecast
quarters are considerably better grounded than the later ones, and an auditor asking how much
of a five-year projection rests on real data has a precise answer.

### Reading the forecast path characteristics

Table 6.4 quantifies what the forecast figures show. The volatility ratio divides each
forecast's average quarter-to-quarter movement by the actual series' average movement over
2015 to 2019, a comparably stable period. A ratio near one means the forecast moves like the
series historically has. A ratio well above one means the model is transmitting quarterly
variation in the regressors into a series that does not historically move that way.

The jump column reports the gap between the last observed value and the first forecast
quarter. Because no model in this notebook carries a lagged dependent term, nothing constrains
the forecast to connect to the last observation, and a large jump is a structural consequence
of that design choice rather than an economic signal.

Two model families fail in a distinct way. Tree ensembles and radial basis function kernels
cannot predict outside the range of their training data. Their forecasts are therefore bounded
by construction, and a flat or capped path from those models reflects model structure rather
than a projection about the economy.

### The final diagnostic

The last cell decomposes the sharpest single feature of the forecast path into each
regressor's contribution, by multiplying the change in each regressor between two consecutive
forecast quarters by its coefficient. Because the contributions sum to the total change, this
identifies exactly which input is responsible.

This is more useful than describing the path as volatile in general terms. It locates the
cause in one variable, which converts a caveat into a specific recommendation about how the
specification could be improved.

### Export

Three files are written to `Stage1_Outputs` for the joint comparison in `03c`: the forecast
paths for every model and robustness run, the full metrics table, and a metadata record so
that a stale output file cannot be read without its provenance. Columns are prefixed `ML_`.
The time-series track writes the equivalent `TS_` file on the same index.

In [41]:
# 6.1 - production forecast, OLS refit on full history
# The evaluation fit used 1991-2020. The production model refits on all history to 2025 Q4
# so the forecast uses every available observation, then projects 2026 Q1 to 2030 Q4 from
# the Stage A macro paths.
ols_prod = results['OLS']['prod_model']
fcst_ols = pd.Series(results['OLS']['fcst_pred'],
                     index=pd.DatetimeIndex(results['OLS']['idx_fcst']),
                     name='OLS')

n_prod = int(ols_prod.nobs)
print(f'Production OLS: n={n_prod} | R2={ols_prod.rsquared:.3f} | '
      f'DW={durbin_watson(ols_prod.resid):.3f}')
print(f'\nForecast 2026 Q1 to 2030 Q4:')
print(f'  range     {fcst_ols.min():.3f}% to {fcst_ols.max():.3f}%')
print(f'  start/end {fcst_ols.iloc[0]:.3f}% -> {fcst_ols.iloc[-1]:.3f}% '
      f'({fcst_ols.iloc[-1] - fcst_ols.iloc[0]:+.3f}pp)')
print(f'  last observed (2025 Q4): {df.loc[EVAL_END, TARGET]:.3f}%')
print(f'  quarter-to-quarter mean absolute change: {fcst_ols.diff().abs().mean():.3f}pp')
print(f'  same for 2015-2019 actuals: '
      f'{df.loc["2015":"2019", TARGET].diff().abs().mean():.3f}pp')

Production OLS: n=137 | R2=0.673 | DW=0.823

Forecast 2026 Q1 to 2030 Q4:
  range     2.247% to 3.595%
  start/end 3.164% -> 3.231% (+0.067pp)
  last observed (2025 Q4): 2.940%
  quarter-to-quarter mean absolute change: 0.224pp
  same for 2015-2019 actuals: 0.034pp


### What the challengers rely on

SHAP attribution decomposes each model's predictions into contributions from the individual
regressors, allowing the two leading challengers to be compared against ordinary least
squares on the same basis. The question is whether an accurate challenger uses the same
economic information in a more flexible way, or relies on a different structure altogether.
The first case leaves the linear coefficients a fair description of what drives the forecast;
the second would mean the linear model is not merely less accurate but is describing a
different relationship.

Two challengers are explained rather than one. A single comparison cannot separate a genuine
departure from the linear specification from an idiosyncrasy of one fitted model, and the
agreement between the two challengers provides the control.

#### Kernel ridge regression reproduces the linear structure almost exactly

Kernel ridge regression agrees with ordinary least squares at a Spearman rank correlation of
0.976. The ordering is identical for the six most important variables and differs only in
transposing the two least important, industrial production and the COVID dummy. The
importance shares are close throughout: credit growth 0.288 against 0.278, consumer
confidence 0.121 against 0.140, house price growth 0.149 against 0.153. The only material
reallocation is toward unemployment, at 0.179 against 0.153.

This holds despite a radial basis function kernel, a genuinely nonlinear form. A nonlinear
model applied to this data recovers the linear model's own importance structure, which
indicates that whatever accuracy difference exists between them arises from the functional
form applied to the same relationships rather than from access to different information.

#### The perceptron departs, and is the outlier of the three

The multi-layer perceptron agrees with ordinary least squares at 0.762. The control figure
makes this interpretable: the two challengers agree with each other at 0.786, barely more
than the perceptron agrees with the linear model, and far less than the 0.976 between kernel
ridge regression and the linear model. The challengers do not form a coherent group departing
from the linear specification in a common direction. The perceptron is the outlier.

Its divergence is concentrated. It reallocates toward house price growth, raising it from a
0.153 share to 0.228 and from third to second in rank, and toward CPI inflation, from 0.078
to 0.114. It reduces both real activity measures, taking industrial production from 0.044 to
0.013 and GDP growth from 0.112 to 0.004, effectively discarding the latter.

#### The GDP result is suggestive but not conclusive

Section 2 identified GDP growth as a probable collinear suppressor: it enters with a sign
contrary to its economic prior, reverses sign as the other real activity measures are added
on a fixed sample, and attains significance only in the full specification.

The perceptron's treatment is consistent with that reading. A model not constrained to a
linear functional form has no requirement for a suppressor variable, and the perceptron does
not retain one.

Kernel ridge regression, however, is also unconstrained by linearity, and it keeps GDP growth
at a 0.100 share, close to the 0.112 of ordinary least squares. Its per-quarter contributions
show the same positive relationship, with low values of GDP growth contributing negatively
and high values positively. It therefore reproduces the suppressor rather than seeing through
it.

Two of the three models retain the variable and one discards it, and the one that discards it
is the model that agrees least with the other two. The SHAP evidence therefore supports the
Section 2 conclusion without independently establishing it. The nested specification trace
remains the stronger evidence, and the attribution should be reported as consistent with it
rather than as confirmation.

#### Both challengers produce monotone effects

Figure 5.3 plots each model's per-quarter contributions, coloured by whether the variable was
high or low in that quarter. Across both models, every variable with meaningful spread shows
a clean colour gradient from one side of zero to the other. Low credit growth contributes
negatively and high credit growth positively; high house price growth contributes negatively
and low values positively; the same clean separation holds for unemployment, consumer
confidence and CPI inflation.

There is no variable in either panel where high and low values appear on the same side of
zero, which is the pattern that would indicate a non-monotone or interaction effect. Both
challengers apply directionally the same relationships a linear model would, and neither has
found a structurally different response.

The COVID dummy behaves as a binary variable should in both panels, with a small number of
quarters receiving a large negative contribution and the remainder a small positive offset.
It confirms the control is functioning and carries no further interpretation.

Kernel ridge regression makes larger per-quarter attributions than the perceptron throughout,
with credit growth contributions spanning roughly minus 1.7 to plus 1.3 percentage points
against minus 1.0 to plus 0.8 for the perceptron. This is consistent with the perceptron
being the more heavily shrunk model and with the forecast volatility results reported below.

#### Implication for the deliverable

The evidence supports treating the ordinary least squares coefficients as a reasonable
description of what drives the forecast. The closest challenger by importance structure
reproduces the linear ordering almost exactly, both challengers produce monotone effects in
the same directions, and neither relies on information the linear specification is unable to
represent. The accuracy differences between these models do not arise from the challengers
identifying structure the linear model misses.

This qualifies rather than removes the caveat on interpretation. Individual partial effects
remain unidentified where regressors share a transmission channel, and SHAP does not resolve
this either: attribution among correlated variables is not uniquely determined, since how
importance is divided between two variables carrying overlapping information depends on the
background distribution used. The three real activity measures should be read as a group
rather than individually, in both the coefficient table and the attribution.

### Instability of challenger selection

No challenger significantly outperformed naive persistence on the clean evaluation window.
More informative than that null result is the finding that the ranking among challengers is
not stable under choices that should be immaterial.

Two perturbations demonstrate this. The ranking depends on the accuracy metric, since RMSE
penalises occasional large errors more heavily than frequent small ones and reorders several
models relative to MAE. It also depends on the criterion used to select hyperparameters:
kernel ridge regression achieves a test RMSE of 0.635 when tuned on validation MAE and 0.765
when tuned on validation RMSE, a deterioration of twenty per cent from the selection rule
alone, holding model class, grid, training data and evaluation window fixed. The two criteria
differ only in the regularisation strength they select, 0.01 against 0.1, with the kernel
width unchanged. Under the first criterion the model beats naive persistence at a skill of
0.103; under the second it does not, at minus 0.080. Neither criterion is the correct one.
The point is that the data cannot distinguish between them.

The cause is the inner validation window. Hyperparameters are chosen by scoring on 2017 Q1
to 2020 Q4, sixteen quarters during which the delinquency rate was both historically low and
unusually flat. A near-constant prediction scores well on a near-constant window, so
validation error partly measures how little a model moves rather than how well it forecasts.

The models that do outperform the benchmark appear to win by damping rather than by
capturing structure. The multi-layer perceptron is the only model whose training error
exceeds its test error, 0.943 against 0.646, which indicates underfitting: a large penalty
prevents it fitting the volatile training period, and the resulting near-flat prediction
suits a flat evaluation window. Its forecast moves 3.3 times as much from quarter to quarter
as the delinquency rate historically has, against 6.7 times for OLS. An unemployment-only
regression, with one macroeconomic variable, achieves a test RMSE of 0.742 and outperforms
the seven-variable specification and all four linear models.

The SHAP attribution supports the same reading. Kernel ridge regression reproduces the OLS
importance ordering almost exactly, at a rank correlation of 0.98, so it applies a nonlinear
form to the same structure rather than exploiting information the linear model cannot reach.
The perceptron departs further, at 0.76, and its departure is concentrated in GDP growth,
to which it assigns a 0.004 share of importance against 0.112 for OLS. That variable was
independently identified in Section 2 as entering with a sign contrary to its economic prior
and reaching significance only in the full specification, consistent with a collinear
suppressor rather than an economic signal. A model not constrained to linearity discards it.

The evidence does not support identifying a best-performing challenger. Fourteen quarters
from a single origin lack the power to separate these models, the ranking moves under
defensible variations in metric and tuning rule, and the apparent gains come from smoothness
rather than flexibility. OLS remains the deliverable as pre-registered, and the improvements
available here lie in the specification, through a smoothed regressor or a lagged dependent
term, rather than in the choice of model class.

In [42]:
# 6.2 - Table 6.1 Equation 5.11 coefficients
eq511 = pd.DataFrame({
    'Coefficient': ols_prod.params.round(4),
    'Std error (NW)': ols_prod.bse.round(4),
    't': ols_prod.tvalues.round(3),
    'p': ols_prod.pvalues.round(4),
}, index=['const'] + FEATURES_EQ511)

eq511['Prior'] = ['n/a'] + [SIGN_PRIORS.get(f, ('n/a',))[0] for f in FEATURES_EQ511]
eq511['Matches'] = [
    'n/a' if pr in ('n/a', '?') else ('Yes' if (b > 0) == (pr == '+') else 'NO')
    for b, pr in zip(eq511['Coefficient'], eq511['Prior'])]

print('Table 6.1 - Equation 5.11, production OLS (Newey-West HAC, maxlags=4)')
print(f'Sample 1991 Q1 to 2025 Q4 | n={n_prod} | R2={ols_prod.rsquared:.3f} | '
      f'adj R2={ols_prod.rsquared_adj:.3f}')
display(eq511)

# Stage A validation status carries into the forecast: a regressor whose own Stage A
# forecast failed the DM test supplies uncertain inputs from 2026 onward.
flagged = [f for f in FEATURES_EQ511
           if any(f.startswith(u) for u in UNVALIDATED_STAGE_A)]
print(f'Regressors without Stage A DM validation: {flagged}')

Table 6.1 - Equation 5.11, production OLS (Newey-West HAC, maxlags=4)
Sample 1991 Q1 to 2025 Q4 | n=137 | R2=0.673 | adj R2=0.652


,Coefficient,Std error (NW),t,p,Prior,Matches
const,22.3142,7.7759,2.870,0.0041,n/a,n/a
us_credit_qoq_growth_L6,0.8515,0.1002,8.502,0.0000,+,Yes
us_unemployment_L0,0.3110,0.0870,3.576,0.0003,+,Yes
us_house_price_yoy_L3,-0.0905,0.0302,-3.000,0.0027,-,Yes
us_gdp_yoy_growth_L2,0.1151,0.0630,1.829,0.0675,-,NO
us_consumer_confidence_L2,-0.2176,0.0781,-2.787,0.0053,-,Yes
us_cpi_L0,0.0977,0.0763,1.280,0.2006,?,n/a
us_indprod_yoy_L3,-0.0050,0.0348,-0.144,0.8852,-,Yes
covid_dummy,-1.3902,0.3357,-4.142,0.0000,n/a,n/a


Regressors without Stage A DM validation: ['us_credit_qoq_growth_L6', 'us_consumer_confidence_L2', 'us_indprod_yoy_L3']


In [43]:
print(f'\nn obs: {int(ols_prod.nobs)}')
print(f'n_prod variable: {n_prod}')


n obs: 137
n_prod variable: 137


### Interpreting the estimated coefficients

Equation 5.11 is estimated on 137 quarters from 1991 Q1 to 2025 Q4 with Newey-West standard
errors at four lags. It explains 67.3 per cent of the variation in the delinquency rate,
65.2 per cent after adjusting for the number of parameters. Each coefficient is the change
in the delinquency rate, in percentage points, per one-unit change in that regressor at the
stated lag, holding the others constant.

#### The intercept is not an economic quantity

At 22.31 the intercept is far outside the range of the delinquency rate, which lies between
2 and 7 per cent throughout the sample. This is a scaling artefact rather than a prediction.
Consumer confidence enters as an index level near 100 rather than as a growth rate, so its
coefficient of minus 0.2176 contributes approximately minus 21.8 percentage points at
typical values. The intercept exists to offset this, and the two together net to
approximately 0.55 percentage points. The intercept should not be interpreted on its own.

#### Four coefficients carry the signal

Credit growth at lag six is the dominant regressor, with a coefficient of 0.8515 and a t
statistic of 8.5, the largest and most precisely estimated in the model. A one percentage
point increase in quarterly household credit growth is associated with a 0.85 percentage
point increase in the delinquency rate six quarters later. The sign matches the prior: rapid
credit expansion implies rising leverage and lending to progressively weaker borrowers, with
delinquency following as loans season. The magnitude is the largest in the specification and
the six-quarter lag is the longest, which together make this variable the principal driver
of the forecast path.

Unemployment enters contemporaneously at 0.3110 and is significant at the 0.03 per cent
level. A one percentage point rise in the unemployment rate is associated with a 0.31
percentage point rise in delinquency in the same quarter. The contemporaneous timing is
consistent with the product: credit card borrowers who lose income can miss a payment within
weeks, unlike mortgage borrowers who typically have more buffer.

House price growth at lag three enters at minus 0.0905 and is significant at 0.3 per cent. A
ten percentage point fall in annual house price growth is associated with a 0.91 percentage
point rise in delinquency three quarters later. Housing is the largest household asset and
the main source of collateral, so falling prices reduce both the capacity to refinance and
the incentive to protect a credit record.

Consumer confidence at lag two enters at minus 0.2176 and is significant at 0.5 per cent.
The variable is an amplitude-adjusted index centred on 100, so realistic movements are of
the order of a few points rather than tens: a three point fall is associated with a 0.65
percentage point rise in delinquency two quarters later.

#### Three coefficients do not

GDP growth at lag two is marginally significant at 0.068 and carries the wrong sign, entering
positive at 0.1151 against a negative prior. This is discussed separately below and should
not be read as evidence that economic growth raises defaults.

CPI inflation at 0.0977 is not significant at 0.20. This is consistent with the prior, which
recorded the direction as ambiguous because inflation both erodes the real value of existing
debt and erodes real income. The estimate does not resolve which channel dominates.
Recalling that CPI was the marginal survivor of the consensus vote, retained on two of four
nominations and dropped when each method nominated four rather than five, the lack of
significance here is coherent with that.

Industrial production at lag three is effectively zero, with a coefficient of minus 0.0050
against a standard error seven times larger. It nominally matches its negative prior, but a
coefficient statistically indistinguishable from zero cannot be said to match any prior in a
meaningful sense. The variable contributes nothing measurable to the fit.

#### The COVID dummy

At minus 1.3902 the dummy is the second largest coefficient and highly significant. In the
production model it spans 2020 Q1 to 2022 Q2, a ten-quarter window, and is identified from
the full distortion period. Robustness run R1 removes the dummy and finds the forecast path
moves by 3.3 per cent of its level, confirming that the choice of whether to include it has
limited practical consequence for the projection.

In [44]:
# 6.3 - Table 6.2 when each regressor switches to Stage A projections
# A lagged regressor draws on observed data for its first few forecast quarters. This
# records the quarter at which each begins using Stage A projections instead, which is
# where forecast uncertainty rises.
rows = []
for f in FEATURES_EQ511:
    if f == 'covid_dummy':
        continue
    lag    = int(f.rsplit('_L', 1)[1])
    switch = (pd.Timestamp(EVAL_END) + pd.DateOffset(months=3 * (lag + 1)) +
              pd.offsets.QuarterEnd(0))
    n_obs  = sum(fcst_ols.index < switch)
    rows.append({'Regressor': f, 'Lag': lag,
                 'Observed inputs until': (q(switch - pd.offsets.QuarterEnd(1))
                                           if n_obs else 'none'),
                 'Forecast quarters on observed data': n_obs,
                 'On Stage A projections': len(fcst_ols) - n_obs})

switch_df = pd.DataFrame(rows).sort_values('Lag', ascending=False)
print('Table 6.2 - Transition from observed to projected regressor inputs')
display(switch_df.set_index('Regressor'))

Table 6.2 - Transition from observed to projected regressor inputs


,Lag,Observed inputs until,Forecast quarters on observed data,On Stage A projections
Regressor,,,,
us_credit_qoq_growth_L6,6,2027 Q2,6,14
us_house_price_yoy_L3,3,2026 Q3,3,17
us_indprod_yoy_L3,3,2026 Q3,3,17
us_consumer_confidence_L2,2,2026 Q2,2,18
us_gdp_yoy_growth_L2,2,2026 Q2,2,18
us_unemployment_L0,0,none,0,20
us_cpi_L0,0,none,0,20


In [45]:
# 6.4 - Table 6.3 robustness runs
# OLS only, so differences are attributable to the specification change rather than to
# model class. R2 is additionally compared against the primary specification fitted on
# R2's own 107-quarter sample, isolating specification from sample composition.
rob = {'Primary': fit_ols(mats_primary),
       'R1 (no dummy)': fit_ols(mats_r1),
       'R2 (all 13)': fit_ols(mats_r2),
       'R2 sample control': fit_ols(mats_primary_r2sample),
       'R3 (raw target)': fit_ols(mats_r3)}

rob_rows = []
for name, r in rob.items():
    ix = pd.DatetimeIndex(r['idx_eval'])
    m  = (ix >= CLEAN_START) & (ix <= CLEAN_END)
    fc = pd.Series(r['fcst_pred'], index=pd.DatetimeIndex(r['idx_fcst']))
    rob_rows.append({
        'Run': name,
        'Train n': len(r['y_train']),
        'RMSE (clean)': round(rmse(r['y_eval'][m], r['eval_pred'][m]), 4),
        'Skill vs persist': round(skill(r['y_eval'][m], r['eval_pred'][m],
                                        bm_persist[m], 'rmse'), 4),
        '2026 Q1': round(fc.iloc[0], 3),
        '2030 Q4': round(fc.iloc[-1], 3),
        'Max gap vs primary': round(
            np.abs(fc.values - rob['Primary']['fcst_pred']).max(), 3),
    })

rob_df = pd.DataFrame(rob_rows).set_index('Run')
print('Table 6.3 - Robustness runs (OLS only)')
display(rob_df)

# A run diverges materially if its forecast departs from the primary path by more than the
# primary model's own clean-window RMSE. The threshold is pre-registered but lenient, so the
# gap is also expressed relative to the forecast level.
tol = rob_df.loc['Primary', 'RMSE (clean)']
lvl = rob['Primary']['fcst_pred'].mean()
print(f'Pre-registered threshold = primary clean RMSE = {tol:.4f}pp')
print(f'Mean forecast level = {lvl:.3f}%, so the threshold is {tol/lvl:.0%} of it '
      f'and is lenient by construction.')
for name in rob_df.index[1:]:
    gap = rob_df.loc[name, 'Max gap vs primary']
    print(f'  {name:<20} max gap {gap:.3f}pp ({gap/lvl:>5.1%} of level)  '
          f'{"MATERIAL" if gap > tol else "within tolerance"}')

Table 6.3 - Robustness runs (OLS only)


,Train n,RMSE (clean),Skill vs persist,2026 Q1,2030 Q4,Max gap vs primary
Run,,,,,,
Primary,117,1.1213,-0.5839,3.164,3.231,0.000
R1 (no dummy),117,1.1576,-0.6351,3.186,3.209,0.109
R2 (all 13),107,1.2037,-0.7002,3.213,3.178,0.167
R2 sample control,107,1.0327,-0.4587,3.124,3.193,0.054
R3 (raw target),113,0.9820,-0.3871,2.955,2.901,0.491


Pre-registered threshold = primary clean RMSE = 1.1213pp
Mean forecast level = 3.278%, so the threshold is 34% of it and is lenient by construction.
  R1 (no dummy)        max gap 0.109pp ( 3.3% of level)  within tolerance
  R2 (all 13)          max gap 0.167pp ( 5.1% of level)  within tolerance
  R2 sample control    max gap 0.054pp ( 1.6% of level)  within tolerance
  R3 (raw target)      max gap 0.491pp (15.0% of level)  within tolerance


In [46]:
# 6.5 - Figure 6.1 production forecasts, one panel per model
# Each model refitted on the full 137-quarter history and projected forward on the Stage A
# macro paths. Tree ensembles and RBF kernels cannot predict outside the range of their
# training data, so flat or bounded paths reflect model structure rather than economics.
hist = df[TARGET].loc[:EVAL_END].dropna()
link = lambda s: ([hist.index[-1]] + list(s.index), [hist.iloc[-1]] + list(s.values))

fcst_all = {n: pd.Series(results[n]['fcst_pred'],
                         index=pd.DatetimeIndex(results[n]['idx_fcst']))
            for n in MODEL_NAMES}

recent   = hist.loc['2018':]
last_obs = df.loc[EVAL_END, TARGET]

ncols = 3
nrows = int(np.ceil(len(MODEL_NAMES) / ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=MODEL_NAMES,
                    vertical_spacing=0.11, horizontal_spacing=0.06,
                    shared_yaxes=True)

for i, name in enumerate(MODEL_NAMES):
    r, c = i // ncols + 1, i % ncols + 1
    xf, yf = link(fcst_all[name])
    colour = TEAL if name == 'OLS' else AMBER

    fig.add_vrect(x0=FCST_START, x1=FCST_END, row=r, col=c,
                  fillcolor='rgba(23,165,137,0.05)', line_width=0)
    fig.add_trace(go.Scatter(x=recent.index, y=recent.values, mode='lines', name='Actual',
                             line=dict(color=NAVY, width=2),
                             showlegend=(i == 0)), row=r, col=c)
    fig.add_trace(go.Scatter(x=xf, y=yf, mode='lines', name='Forecast',
                             line=dict(color=colour, width=1.8),
                             showlegend=(i == 0)), row=r, col=c)
    fig.add_hline(y=last_obs, line_dash='dot', line_color=GREY, line_width=1,
                  row=r, col=c)

fig.update_layout(
    title=dict(text=('<b>Figure 6.1 - Production Forecasts to 2030 Q4</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     f'Dotted line = last observed value, {last_obs:.2f}% in 2025 Q4. '
                     'Shaded band = forecast horizon. OLS is the primary model.</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.975, yanchor='top'),
    template=TEMPLATE, height=250 * nrows,
    legend=dict(orientation='h', yanchor='bottom', y=1.03, xanchor='center', x=0.5),
    margin=dict(t=130, b=45))
fig.update_yaxes(title_text='Delinquency Rate (%)', col=1)
fig.show()

In [47]:
# 6.6 - Table 6.4 forecast path characteristics
# Quantifies what Figure 6.1 shows. The volatility ratio compares each forecast's
# quarter-to-quarter movement against the actual series over 2015-2019, a comparably
# stable period. A ratio far above 1 means the model transmits macro noise into a series
# that does not historically move that way.
hist_vol = df.loc['2015':'2019', TARGET].diff().abs().mean()

fc_rows = []
for n in MODEL_NAMES:
    s = fcst_all[n]
    fc_rows.append({
        'Model': n,
        '2026 Q1': round(s.iloc[0], 3),
        '2030 Q4': round(s.iloc[-1], 3),
        'Min': round(s.min(), 3), 'Max': round(s.max(), 3),
        'Range': round(s.max() - s.min(), 3),
        'Q/Q change': round(s.diff().abs().mean(), 3),
        'Vol ratio': round(s.diff().abs().mean() / hist_vol, 1),
        'Jump at 2026 Q1': round(s.iloc[0] - last_obs, 3),
    })

fcst_summary = pd.DataFrame(fc_rows).set_index('Model')
print('Table 6.4 - Forecast path characteristics')
print(f'Last observed 2025 Q4 = {last_obs:.3f}% | '
      f'actual quarter-to-quarter change 2015-2019 = {hist_vol:.3f}pp')
display(fcst_summary)

print('Vol ratio is forecast quarter-to-quarter movement divided by the 2015-2019 actual.')
print('A discontinuity at 2026 Q1 indicates the forecast does not connect to the last '
      'observed value, since no model carries a lagged dependent term.')

Table 6.4 - Forecast path characteristics
Last observed 2025 Q4 = 2.940% | actual quarter-to-quarter change 2015-2019 = 0.034pp


,2026 Q1,2030 Q4,Min,Max,Range,Q/Q change,Vol ratio,Jump at 2026 Q1
Model,,,,,,,,
OLS,3.164,3.231,2.247,3.595,1.348,0.224,6.7,0.224
Ridge,3.177,3.242,2.276,3.600,1.324,0.221,6.6,0.237
Lasso,3.184,3.254,2.315,3.596,1.281,0.215,6.4,0.244
Elastic Net,3.184,3.254,2.315,3.596,1.281,0.215,6.4,0.244
KRR,3.155,3.200,2.233,3.597,1.364,0.223,6.6,0.215
SVR,3.189,3.277,3.182,3.446,0.263,0.058,1.7,0.249
Random Forest,3.788,2.977,2.977,4.260,1.283,0.127,3.8,0.848
XGBoost,3.638,2.772,2.772,4.087,1.315,0.137,4.1,0.698
MLP,3.145,3.159,2.715,3.358,0.643,0.113,3.3,0.205


Vol ratio is forecast quarter-to-quarter movement divided by the 2015-2019 actual.
A discontinuity at 2026 Q1 indicates the forecast does not connect to the last observed value, since no model carries a lagged dependent term.


In [48]:
'''
# 6.7 - Figure 6.2 robustness runs
fig = go.Figure()

recent_r = hist.loc['2019':]
fig.add_trace(go.Scatter(x=recent_r.index, y=recent_r.values, mode='lines', name='Actual',
                         line=dict(color=NAVY, width=2.4)))
for (name, r), colr in zip(rob.items(), [TEAL, AMBER, BLUE, GREEN, RED]):
    s = pd.Series(r['fcst_pred'], index=pd.DatetimeIndex(r['idx_fcst']))
    xr, yr = link(s)
    fig.add_trace(go.Scatter(x=xr, y=yr, mode='lines', name=name,
                             line=dict(color=colr, width=2.2 if name == 'Primary' else 1.4,
                                       dash='solid' if name == 'Primary' else 'solid')))

fig.update_layout(
    title=dict(text=('<b>Figure 6.2 - Robustness Runs Over the Forecast Horizon</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     'OLS under four alternative specifications, each changing one '
                     'modelling choice relative to the primary run</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.95, yanchor='top'),
    xaxis_title='Quarter', yaxis_title='Delinquency Rate (%)',
    template=TEMPLATE, height=430,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(t=125, b=50))
fig.show()
'''

'\n# 6.7 - Figure 6.2 robustness runs\nfig = go.Figure()\n\nrecent_r = hist.loc[\'2019\':]\nfig.add_trace(go.Scatter(x=recent_r.index, y=recent_r.values, mode=\'lines\', name=\'Actual\',\n                         line=dict(color=NAVY, width=2.4)))\nfor (name, r), colr in zip(rob.items(), [TEAL, AMBER, BLUE, GREEN, RED]):\n    s = pd.Series(r[\'fcst_pred\'], index=pd.DatetimeIndex(r[\'idx_fcst\']))\n    xr, yr = link(s)\n    fig.add_trace(go.Scatter(x=xr, y=yr, mode=\'lines\', name=name,\n                             line=dict(color=colr, width=2.2 if name == \'Primary\' else 1.4,\n                                       dash=\'solid\' if name == \'Primary\' else \'solid\')))\n\nfig.update_layout(\n    title=dict(text=(\'<b>Figure 6.2 - Robustness Runs Over the Forecast Horizon</b>\'\n                     f\'<br><span style="font-size:11.5px;color:{GREY}">\'\n                     \'OLS under four alternative specifications, each changing one \'\n                     \'modelling choice re

In [49]:
# 6.7 - Figure 6.2 robustness runs
fig = go.Figure()

recent_r = hist.loc['2019':]
fig.add_trace(go.Scatter(x=recent_r.index, y=recent_r.values, mode='lines', name='Actual',
                         line=dict(color=NAVY, width=2.4)))
for (name, r), colr in zip(rob.items(), [TEAL, AMBER, BLUE, GREEN, RED]):
    if name == 'R2 sample control':
        continue
    s = pd.Series(r['fcst_pred'], index=pd.DatetimeIndex(r['idx_fcst']))
    xr, yr = link(s)
    fig.add_trace(go.Scatter(x=xr, y=yr, mode='lines', name=name,
                             line=dict(color=colr, width=2.2 if name == 'Primary' else 1.4,
                                       dash='solid' if name == 'Primary' else 'solid')))

fig.update_layout(
    title=dict(text=('<b>Figure 6.2 - Robustness Runs Over the Forecast Horizon</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     'OLS under four alternative specifications, each changing one '
                     'modelling choice relative to the primary run</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.95, yanchor='top'),
    xaxis_title='Quarter', yaxis_title='Delinquency Rate (%)',
    template=TEMPLATE, height=430,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(t=125, b=50))
fig.show()

In [50]:
# 6.8 - decomposing the sharpest forecast movement
# Every specification, including R3, drops sharply at 2026 Q2 and recovers. The cause must
# be in the regressor inputs rather than the COVID treatment. This decomposes the change
# in each regressor's contribution between 2026 Q1 and Q2.
beta_p = dict(zip(FEATURES_EQ511, ols_prod.params[1:]))
q1, q2 = fcst_ols.index[0], fcst_ols.index[1]

dec = pd.DataFrame([{
    'Regressor': f,
    '2026 Q1 value': round(df.loc[q1, f], 4),
    '2026 Q2 value': round(df.loc[q2, f], 4),
    'Change': round(df.loc[q2, f] - df.loc[q1, f], 4),
    'Coefficient': round(beta_p[f], 4),
    'Contribution to dip': round((df.loc[q2, f] - df.loc[q1, f]) * beta_p[f], 4),
} for f in FEATURES_EQ511]).sort_values('Contribution to dip')

print(f'Forecast change 2026 Q1 to Q2: '
      f'{fcst_ols.iloc[1] - fcst_ols.iloc[0]:+.3f}pp')
display(dec.set_index('Regressor'))
print(f'Sum of contributions: {dec["Contribution to dip"].sum():+.3f}pp')

Forecast change 2026 Q1 to Q2: -0.917pp


,2026 Q1 value,2026 Q2 value,Change,Coefficient,Contribution to dip
Regressor,,,,,
us_credit_qoq_growth_L6,0.8295,-0.2092,-1.0387,0.8515,-0.8845
us_consumer_confidence_L2,99.8105,100.2648,0.4543,-0.2176,-0.0989
us_gdp_yoy_growth_L2,2.3352,1.9893,-0.3459,0.1151,-0.0398
us_indprod_yoy_L3,0.5793,1.8632,1.2838,-0.0050,-0.0064
covid_dummy,0.0000,0.0000,0.0000,-1.3902,-0.0000
us_cpi_L0,2.8717,2.8760,0.0043,0.0977,0.0004
us_unemployment_L0,4.4278,4.4985,0.0707,0.3110,0.0220
us_house_price_yoy_L3,0.6631,-0.3358,-0.9990,-0.0905,0.0904


Sum of contributions: -0.917pp


In [51]:
# 6.9 - Figure 6.3 equal-weighted combination of the linear model and two challengers
# Equal weights rather than performance weights. Estimating weights from the evaluation
# window and then scoring the combination on that same window is circular, and the
# validation window has already been shown unable to distinguish these models reliably. In
# small samples equal weights also tend to outperform estimated optimal weights, since the
# estimation error in the weights exceeds the theoretical gain from optimising them.
ENS_MEMBERS = ['OLS', best_ml, second_ml]
W           = 1.0 / len(ENS_MEMBERS)

ens_fcst = sum(fcst_all[m] * W for m in ENS_MEMBERS)
ens_eval = sum(results[m]['eval_pred'] * W for m in ENS_MEMBERS)

fig = go.Figure()
recent_e = hist.loc['2019':]
fig.add_trace(go.Scatter(x=recent_e.index, y=recent_e.values, mode='lines', name='Actual',
                         line=dict(color=NAVY, width=2.4)))

series = [('OLS (primary)', fcst_all['OLS'], TEAL, 'solid', 1.6),
          (best_ml, fcst_all[best_ml], AMBER, 'solid', 1.6),
          (second_ml, fcst_all[second_ml], BLUE, 'solid', 1.6),
          ('Equal-weighted average', ens_fcst, RED, 'solid', 2)]

for label, s, colr, dash, wid in series:
    xr, yr = link(s)
    fig.add_trace(go.Scatter(x=xr, y=yr, mode='lines', name=label,
                             line=dict(color=colr, width=wid, dash=dash)))

fig.add_hline(y=last_obs, line_dash='dot', line_color=GREY, line_width=1)

fig.update_layout(
    title=dict(text=('<b>Figure 6.3 - Equal-Weighted Combination Against Its Members</b>'
                     f'<br><span style="font-size:11.5px;color:{GREY}">'
                     f'One third each on OLS, {best_ml} and {second_ml}. Dotted line is '
                     f'the last observed value, {last_obs:.2f} per cent in 2025 Q4</span>'),
               font=dict(size=16, color=NAVY), x=0.015, xanchor='left',
               y=0.95, yanchor='top'),
    xaxis_title='Quarter', yaxis_title='Delinquency Rate (%)',
    template=TEMPLATE, height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(t=130, b=50))
fig.show()

In [52]:
# 6.10 - Table 6.5 does combination help
# Two questions. Does the combination forecast more accurately on the evaluation window
# than its members, and does it move less from quarter to quarter in production. Weights
# were fixed at one third without reference to either window, so the evaluation figures
# below are an out-of-sample result rather than a fitted one.
p_clean = bm_persist[clean_mask]

rows = []
for label, ev, fc in ([(m, results[m]['eval_pred'], fcst_all[m]) for m in ENS_MEMBERS]
                      + [('Equal-weighted', ens_eval, ens_fcst),
                         ('Naive persistence', bm_persist, None)]):
    r = {'Forecast': label,
         'Clean RMSE': round(rmse(y_ev[clean_mask], ev[clean_mask]), 4),
         'Clean MAE': round(mae(y_ev[clean_mask], ev[clean_mask]), 4),
         'Skill vs persist': round(skill(y_ev[clean_mask], ev[clean_mask],
                                         p_clean, 'rmse'), 4)}
    if fc is not None:
        r['2026 Q1'] = round(fc.iloc[0], 3)
        r['2030 Q4'] = round(fc.iloc[-1], 3)
        r['Q/Q change'] = round(fc.diff().abs().mean(), 3)
        r['Vol ratio'] = round(fc.diff().abs().mean() / hist_vol, 1)
    rows.append(r)

ens_df = pd.DataFrame(rows).set_index('Forecast')
print('Table 6.5 - Equal-weighted combination against its members')
print(f'Volatility ratio is against the 2015-2019 actual of {hist_vol:.3f}pp per quarter.')
display(ens_df)

best_member = min(ENS_MEMBERS,
                  key=lambda m: rmse(y_ev[clean_mask],
                                     results[m]['eval_pred'][clean_mask]))
ens_r  = ens_df.loc['Equal-weighted', 'Clean RMSE']
best_r = ens_df.loc[best_member, 'Clean RMSE']
print(f'Combination RMSE {ens_r:.4f} against best member {best_member} at {best_r:.4f}: '
      f'{"combination wins" if ens_r < best_r else "best member wins"}.')
print('The combination is exploratory. OLS alone remains the pre-registered deliverable.')

Table 6.5 - Equal-weighted combination against its members
Volatility ratio is against the 2015-2019 actual of 0.034pp per quarter.


,Clean RMSE,Clean MAE,Skill vs persist,2026 Q1,2030 Q4,Q/Q change,Vol ratio
Forecast,,,,,,,
OLS,1.1213,0.8566,-0.5839,3.164,3.231,0.224,6.7
MLP,0.6464,0.5079,0.0870,3.145,3.159,0.113,3.3
KRR,0.7647,0.6155,-0.0802,3.155,3.200,0.223,6.6
Equal-weighted,0.8337,0.6576,-0.1776,3.155,3.196,0.186,5.5
Naive persistence,0.7080,0.6451,0.0000,NaN,NaN,NaN,NaN


Combination RMSE 0.8337 against best member MLP at 0.6464: best member wins.
The combination is exploratory. OLS alone remains the pre-registered deliverable.


In [53]:
# 6.11 - export for 03c
OUT_DIR.mkdir(parents=True, exist_ok=True)

fcst_out = pd.DataFrame({'ML_OLS_primary': fcst_ols})
ROB_KEYS = {'R1 (no dummy)': 'R1_nodummy', 'R2 (all 13)': 'R2_all13',
            'R2 sample control': 'R2_samplectrl', 'R3 (raw target)': 'R3_rawtarget'}
for name, r in rob.items():
    if name != 'Primary':
        fcst_out[f'ML_OLS_{ROB_KEYS[name]}'] = pd.Series(
            r['fcst_pred'], index=pd.DatetimeIndex(r['idx_fcst']))
for name in MODEL_NAMES:
    if name != 'OLS':
        fcst_out[f'ML_{name.replace(" ", "_")}'] = fcst_all[name]

# Metadata travels with the outputs so a stale file cannot be read without provenance.
# The protocol keys matter for 03c: the two Stage B tracks share a scoring window but
# not a forecast protocol, so RMSE is not comparable across them and skill against each
# track's own naive baseline is.
meta = pd.DataFrame({
    'key': ['notebook', 'target', 'target_variant', 'protocol', 'origin',
            'train_n', 'production_n', 'features', 'primary_model',
            'clean_window', 'generated'],
    'value': ['03b_Stage2_ML', TARGET,
              'spline-adjusted, covid_dummy in specification',
              'single origin, h=1..20, no refit, no realised target after origin',
              '2020 Q4', len(mats_primary['train'][1]), n_prod,
              ' '.join(FEATURES_EQ511), 'OLS',
              f'{CLEAN_START} to {CLEAN_END}',
              pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')],
})

fcst_out.round(4).to_csv(OUT_DIR / 'ML_PD_forecasts_US_Q.csv')
metrics_df.to_csv(OUT_DIR / 'ML_metrics_US_Q.csv', index=False)
meta.to_csv(OUT_DIR / 'ML_run_metadata_US_Q.csv', index=False)

print(f'Written to {OUT_DIR}:')
print(f'  ML_PD_forecasts_US_Q.csv   {fcst_out.shape[0]} rows x {fcst_out.shape[1]} cols')
print(f'  ML_metrics_US_Q.csv        {metrics_df.shape[0]} rows')
print(f'  ML_run_metadata_US_Q.csv   {len(meta)} keys')
print('\nFor 03c: the time-series track writes TS_PD_forecasts_US_Q.csv on the same '
      'index (2026 Q1 to 2030 Q4) with a TS_ column prefix.')

Written to c:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Stage2_Outputs:
  ML_PD_forecasts_US_Q.csv   20 rows x 13 cols
  ML_metrics_US_Q.csv        24 rows
  ML_run_metadata_US_Q.csv   11 keys

For 03c: the time-series track writes TS_PD_forecasts_US_Q.csv on the same index (2026 Q1 to 2030 Q4) with a TS_ column prefix.


## APPENDIX - OLS Model Equation and Coefficient Interpretation

#### Equation 5.11

The consensus vote retained seven macroeconomic regressors plus the COVID structural
break dummy. The fitted OLS equation on the full 140-quarter panel is:

Delinquency(t) = 22.314
                 + 0.852 * CREDIT(t-6)
                 + 0.311 * UNEMP(t)
                 - 0.091 * HPI(t-3)
                 + 0.115 * GDP(t-2)
                 - 0.218 * CC(t-2)
                 + 0.098 * CPI(t)
                 - 0.005 * INDPROD(t-3)
                 - 1.390 * COVID_DUMMY(t)

where all variables are defined at the lags shown, standard errors are
Newey-West HAC with maxlags=4, and the sample runs from 1991 Q1 to 2025 Q4
(N=137, R2=0.673, Adjusted R2=0.652, DW=0.823).

---

#### Coefficient interpretation

**Intercept: 22.314 (p=0.004)**

The intercept represents the baseline delinquency rate when all regressors are at
zero. It has no direct economic interpretation since the regressors never take a
value of zero simultaneously, but its significance confirms the model is correctly
centred. A value of 22% is not a meaningful standalone prediction - it simply absorbs
the unconditional level of the series.

---

**Credit QoQ growth at lag 6: +0.852 (p<0.001) - SIGN MATCHES PRIOR**

The strongest and most precisely estimated coefficient in the model. A one-percentage-
point rise in quarterly credit growth today is associated with a 0.852 percentage point
rise in the delinquency rate six quarters (18 months) later. This matches the expected
positive sign. The transmission mechanism is credit overextension: rapid credit growth
indicates rising leverage in the household sector, which takes roughly 18 months to
manifest as missed payments as borrowers become overextended and income shocks arrive.
This is the longest lag in the model and its significance confirms that credit growth
is a leading indicator of default stress with a meaningful delay, consistent with the
literature on credit cycles (Djeundje and Crook 2019).

---

**Unemployment rate at lag 0: +0.311 (p<0.001) - SIGN MATCHES PRIOR**

The second most significant regressor. A one-percentage-point rise in the unemployment
rate in the same quarter is associated with a 0.311 percentage point rise in the
delinquency rate. The expected sign is positive since job loss directly impairs a
borrower's ability to make payments. The contemporaneous lag is consistent with the
speed at which job losses translate into missed credit card payments - unlike mortgages,
credit card defaults can materialise within weeks of income loss. The coefficient is
economically meaningful: moving from 4% to 8% unemployment (as in the GFC) would
add approximately 1.2 percentage points to the predicted delinquency rate, all else equal.

---

**House price YoY growth at lag 3: -0.091 (p=0.003) - SIGN MATCHES PRIOR**

A one-percentage-point rise in house prices three quarters earlier is associated with
a 0.091 percentage point fall in the delinquency rate. The expected sign is negative
because rising house prices strengthen household balance sheets: borrowers who can
refinance against rising collateral are less likely to default on unsecured credit
obligations. The three-quarter lag reflects the time needed for balance sheet effects
to translate into spending and repayment behaviour. The coefficient is smaller than
those for credit growth and unemployment, which is plausible since house prices
affect credit card defaults indirectly through wealth effects rather than directly
through income constraints.

---

**Real GDP YoY growth at lag 2: +0.115 (p=0.068) - SIGN DOES NOT MATCH PRIOR**

This is the most notable anomaly in the model. The expected sign for GDP growth is
negative - stronger economic growth should reduce defaults - but the estimated
coefficient is positive and marginally significant at the 10% level. The most likely
explanation is collinearity with credit growth: periods of rapid economic expansion
were often accompanied by rapid credit growth, and the two variables are jointly
capturing the credit-cycle dynamic where growth-fuelled borrowing six to eight
quarters earlier leads to elevated defaults later. When credit growth is controlled
for, the incremental effect of GDP growth on defaults becomes ambiguous. The
coefficient is not significant at 5% and should be treated with caution. This finding
is worth discussing in the report as it illustrates the risk Bellotti and Crook (2013)
identify: a variable that behaves as expected univariately can exhibit unexpected
behaviour in a joint model with correlated predictors.

---

**Consumer confidence at lag 2: -0.218 (p=0.005) - SIGN MATCHES PRIOR**

A one-index-point rise in consumer confidence two quarters earlier is associated with
a 0.218 percentage point fall in the delinquency rate. The expected sign is negative
since rising confidence indicates that households feel financially secure and are
more likely to manage their debt obligations. Consumer confidence is a leading
indicator - it shifts before actual spending and repayment behaviour changes - which
explains the two-quarter lag. The coefficient is economically meaningful and precisely
estimated.

---

**CPI YoY inflation at lag 0: +0.098 (p=0.201) - SIGN AMBIGUOUS, NOT SIGNIFICANT**

The expected sign for CPI is theoretically ambiguous. Moderate inflation can erode
the real value of outstanding debt, making it easier to repay, which would suggest
a negative sign. However, high or sustained inflation erodes real household income
and increases debt-servicing costs when interest rates rise in response, which would
suggest a positive sign. The estimated positive coefficient is consistent with the
income-erosion channel dominating, but the variable is not significant at conventional
levels (p=0.201). CPI was included on theoretical grounds despite not clearing the
CCF significance threshold in the EDA, and the joint model confirms it adds limited
incremental explanatory power. It is retained because the inflation channel is
theoretically important for IFRS 9 scenario construction, but its coefficient
should be interpreted cautiously.

---

**Industrial production YoY at lag 3: -0.005 (p=0.885) - SIGN MATCHES PRIOR BUT IRRELEVANT**

The near-zero coefficient and very high p-value indicate that industrial production
adds essentially no information once the other seven variables are included. The
expected sign is negative (production growth signals economic strength), and the
sign is indeed negative, but the magnitude is negligible. This variable survived
the consensus vote - appearing in at least two of the four top-five selection lists -
but its joint significance with the full regressor set is close to zero. This is
precisely the pattern Bellotti and Crook (2013) document: univariate screening can
identify variables that appear informative in isolation but are redundant in a joint
model. Industrial production is highly correlated with GDP growth, and once GDP is
included, industrial production contributes nothing additional. This is a limitation
worth noting in the report.

---

**COVID dummy: -1.390 (p<0.001) - SIGN MATCHES PRIOR**

The COVID structural break dummy takes a value of one during 2020 Q1 to 2022 Q2
and zero otherwise. The large negative coefficient (-1.390 percentage points) confirms
that, even after the spline adjustment to the target variable, a significant residual
downward distortion in the delinquency rate during the COVID period remains unexplained
by the macroeconomic regressors alone. This is consistent with the spline correcting
for the most obvious distortion while the dummy absorbs remaining structural break
effects. The dummy is highly significant and its inclusion improves model fit
substantially - its omission is tested in robustness run R1.

---

#### Overall model assessment

The model explains 67.3% of variance in the delinquency rate over the full sample
(R2=0.673), with an adjusted R2 of 0.652 after penalising for the eight regressors.
Five of the eight regressors (credit growth, unemployment, house prices, consumer
confidence, COVID dummy) are significant at 1%, and all five have the expected sign.
GDP growth is marginally significant with an unexpected positive sign. CPI and
industrial production are not significant in the joint model.

The Durbin-Watson statistic of 0.823 indicates persistent positive serial correlation
in the residuals - forecast errors in one quarter tend to be followed by errors of the
same sign in the next quarter. This is why Newey-West standard errors are used
throughout rather than conventional OLS standard errors. The serial correlation
does not bias the coefficient estimates but does mean the true standard errors are
larger than they would be with independent residuals, and significance conclusions
should be treated accordingly.